# ISIC 2017: Florence-Prefix Captions + Full-Both M-TACS Segmentation

Generate/resume new ISIC captions.csv, encode with frozen CLIP, train the same query/candidate text fusion and
visual+text selector as the Qwen8B run, then compare held-out test Dice/IoU against both historical runs.
This is frozen caption inference, not Florence fine-tuning. No DTD CSV or diagnosis/mask enters the captioner.

**Why Prefix:** the saved DTD split1/seed42 classifier results show macro-F1 0.810651 (Prefix), 0.809382 (FirstValid),
0.790596 (MultiGrounded), 0.779611 (SingleGrounded). Prefix was best among those Florence variants on DTD,
not a proven ISIC winner. This transfer is exploratory after previous test results were observed.

## Offline Inputs and Run Order

1. ISIC 2017 standard image/mask splits: 2,000 train / 150 val / 600 test.
2. The same local DINOv3 code and **ViT-S/16** weights used in the visual and Qwen segmentation experiments.
3. Original **microsoft/Florence-2-base** legacy model folder, including weights, tokenizer, processor and its three
   Python code files. Default: `/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi/Florence-2-base`.
4. Attached `offline_wheels.zip`, `wheels.zip` or wheel folder with Transformers 4.49.0, tokenizers 0.21.x,
   huggingface-hub <1 and required einops/timm/accelerate, compatible with this Kaggle Python/platform.
   These are only needed if a compatible legacy child environment is not already available.
5. Full local CLIP ViT-B/32 checkpoint and tokenizer. **No Qwen checkpoint is needed.**

Run All from the top. Legacy Florence runs in a child process with offline dependencies installed into its own
`worker_runtime/legacy_packages` when needed (`pip --no-index --target`, never an Internet install).
The kernel's Transformers is not downgraded. CLIP runs in a separate child using the kernel environment.
Both workers exit before segmentation initializes; tokenizer parallelism is disabled before imports/workers.

The segmentation recipe is unchanged: 448, frozen DINOv3 ViT-S/16, two final-token gated cross-attention blocks,
DPT-lite, BCE+Dice, training-only 400-image pool, K=1, batch8, seed42, max100 epochs, patience15.
Best validation Dice selects the checkpoint; test does not select prompts, policies or epochs.


In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
import importlib.util
required = ['numpy', 'pandas', 'torch', 'torchvision', 'PIL', 'matplotlib', 'tqdm']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing offline dependencies: ' + ', '.join(missing))
print('Offline mode; tokenizer parallelism disabled before imports/workers.')


In [ ]:
import json
import math
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

PROJECT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_PUT_DIR = Path(os.getenv("ISIC_FLORENCE_PREFIX_OUTPUT", str(PROJECT_DIR / "outputs" / "mtacs_isic2017_448_florence_prefix_full_both")))
OUTPUT_DIR = OUT_PUT_DIR
OUT_PUT_DIR.mkdir(parents=True, exist_ok=True)

# DINOv3 repo/weights: chỉnh lại nếu Kaggle input của bạn đặt khác.
DINOV3_REPO_DIR = "/kaggle/input/datasets/nguyenducxu/dinov3-code/dinov3-main"
if not Path(DINOV3_REPO_DIR).exists():
    for p in [
        Path("/kaggle/input/dinov3-code/dinov3-main"),
        Path("/kaggle/input/dinov3-code"),
        Path("./dinov3-main"),
    ]:
        if p.exists():
            DINOV3_REPO_DIR = str(p)
            break

DINOV3_MODEL_NAME = "dinov3_vits16"
DINOV3_WEIGHTS = "/kaggle/input/datasets/nguyenducxu/dinov3/dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
if not Path(DINOV3_WEIGHTS).exists():
    for p in [
        Path("/kaggle/input/dinov3/dinov3_vits16_pretrain_lvd1689m-08c60483.pth"),
        Path("/kaggle/input/dinov3/dinov3_vits16.pth"),
        Path("./dinov3_vits16_pretrain_lvd1689m-08c60483.pth"),
    ]:
        if p.exists():
            DINOV3_WEIGHTS = str(p)
            break

BACKBONE_NAME = DINOV3_MODEL_NAME
FEATURE_DIM = 384
PATCH_SIZE = 16
IMAGE_SIZE = 448

EPOCHS = 100
BATCH_SIZE = 8
NUM_WORKERS = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 5
PATIENCE = 15
GRAD_CLIP_NORM = 1.0

CANDIDATE_POOL_FRACTION = 0.20
CANDIDATE_FEATURE_BATCH_SIZE = 64
TACS_GUMBEL_TAU = 0.10
TACS_POLICY_WEIGHT = 1.0
TACS_ENTROPY_WEIGHT = 0.01
TACS_SELECTOR_DROPOUT = 0.10
TACS_SELECTOR_TEMPERATURE = 1.0
NUM_CROSS_ATTENTION_LAYERS = 2
CROSS_ATTENTION_HEADS = 6
DECODER_DROPOUT = 0.10

DICE_LOSS_WEIGHT = 1.0
BCE_LOSS_WEIGHT = 1.0
THRESHOLD = 0.5

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print("OUT_PUT_DIR:", OUT_PUT_DIR.resolve())

# Frozen Florence-Prefix, with a separate legacy caption runtime.
ISIC_ROOT = os.getenv("ISIC_ROOT", "")
FLORENCE_MODEL_REF = os.getenv("FLORENCE_MODEL_REF", "/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi/Florence-2-base")
OFFLINE_ASSETS_ROOT = os.getenv("FLORENCE_OFFLINE_ASSETS_ROOT", "/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi")
CLIP_MODEL_LOCAL_PATH = os.getenv("CLIP_MODEL_LOCAL_PATH", "/kaggle/input/datasets/nguyenducxu/vit-32/model_clip_vit")
SEARCH_ROOTS = [Path(OFFLINE_ASSETS_ROOT), Path("/kaggle/input")]
CAPTION_OUTPUT = OUT_PUT_DIR / "caption_generation"
RESUME_TRAINING = True
MAX_NEW_CAPTION_IMAGES = 0  # 8=smoke test; reset to 0 and resume before training.
CAPTION_SETTINGS = {"batch_size": 8, "max_new_tokens": 48, "seed": 42}
if DEVICE.type != "cuda":
    raise RuntimeError("Enable a GPU accelerator before Run All")


In [ ]:
def find_isic_dataset_root():
    manual_candidates = [
        Path("/kaggle/input/datasets/arnabpurk/isic-2017-segmentation-dataset-original/ISIC-2017 Segmentation Dataset - Original"),
        Path("/kaggle/input/isic-2017-segmentation-dataset"),
        Path("/kaggle/input/ISIC-2017 Segmentation Dataset"),
        Path("./ISIC-2017 Segmentation Dataset"),
    ]
    for root in manual_candidates:
        if (root / "train" / "images").is_dir() and (root / "train" / "masks").is_dir():
            return root

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for root in input_root.rglob("*"):
            if not root.is_dir():
                continue
            if (root / "train" / "images").is_dir() and (root / "train" / "masks").is_dir():
                return root

    raise FileNotFoundError(
        "Khong tim thay DATASET_ROOT co train/images va train/masks. "
        "Hay sua DATASET_ROOT thu cong trong cell nay."
    )


DATASET_ROOT = Path(ISIC_ROOT) if ISIC_ROOT else find_isic_dataset_root()
print("DATASET_ROOT:", DATASET_ROOT)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def normalize_mask_stem(stem):
    for suffix in ["_segmentation", "_mask", "_lesion", "_Segmentation", "_Mask"]:
        if stem.endswith(suffix):
            return stem[: -len(suffix)]
    return stem


def list_split_pairs(dataset_root, split):
    image_dir = dataset_root / split / "images"
    mask_dir = dataset_root / split / "masks"
    if not image_dir.is_dir() or not mask_dir.is_dir():
        raise FileNotFoundError(f"Thieu {split}/images hoac {split}/masks trong {dataset_root}")

    image_paths = sorted([p for p in image_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS])
    mask_paths = sorted([p for p in mask_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS])
    mask_index = {normalize_mask_stem(p.stem): p for p in mask_paths}

    rows = []
    missing = []
    for image_path in image_paths:
        key = image_path.stem
        mask_path = mask_index.get(key)
        if mask_path is None:
            missing.append(image_path.name)
            continue
        rows.append({
            "split": split,
            "fname": image_path.name,
            "image_id": image_path.stem,
            "image_path": str(image_path),
            "mask_path": str(mask_path),
        })

    if missing:
        raise FileNotFoundError(f"Khong tim thay mask cho {len(missing)} anh, vi du: {missing[:5]}")
    if not rows:
        raise RuntimeError(f"Khong doc duoc cap image/mask nao trong split {split}")

    df = pd.DataFrame(rows).reset_index(drop=True)
    df["row_id"] = np.arange(len(df))
    return df


train_df = list_split_pairs(DATASET_ROOT, "train")
val_df = list_split_pairs(DATASET_ROOT, "val")
test_df = list_split_pairs(DATASET_ROOT, "test")

candidate_rng = np.random.default_rng(SEED)
candidate_pool_size = max(1, int(round(len(train_df) * CANDIDATE_POOL_FRACTION)))
candidate_pool_positions = np.sort(candidate_rng.choice(len(train_df), size=candidate_pool_size, replace=False))
candidate_df = train_df.iloc[candidate_pool_positions].reset_index(drop=True)


print(f"Train images: {len(train_df):,}")
print(f"Val images: {len(val_df):,}")
print(f"Test images: {len(test_df):,}")
print(f"Candidate pool: {len(candidate_df):,}/{len(train_df):,} train images ({CANDIDATE_POOL_FRACTION:.0%})")
display(train_df.head())
display(candidate_df.head())

all_split_ids = pd.concat([train_df.image_id, val_df.image_id, test_df.image_id]).astype(str)
if all_split_ids.duplicated().any():
    raise ValueError("Duplicate/overlapping image IDs across ISIC train, val and test")
if [len(train_df), len(val_df), len(test_df)] != [2000, 150, 600]:
    raise ValueError("Expected ISIC2017 standard 2000/150/600 image-mask counts; verify ISIC_ROOT")


In [ ]:
class JointTrainTransform:
    def __init__(self, image_size=256):
        self.image_size = int(image_size)
        self.color_jitter = transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.10,
            hue=0.02,
        )

    def __call__(self, image, mask):
        image = image.convert("RGB")
        mask = mask.convert("L")

        i, j, h, w = transforms.RandomResizedCrop.get_params(
            image,
            scale=(0.75, 1.00),
            ratio=(0.90, 1.10),
        )
        image = TF.resized_crop(
            image, i, j, h, w, (self.image_size, self.image_size), InterpolationMode.BILINEAR
        )
        mask = TF.resized_crop(
            mask, i, j, h, w, (self.image_size, self.image_size), InterpolationMode.NEAREST
        )

        if random.random() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)
        if random.random() < 0.5:
            image = TF.vflip(image)
            mask = TF.vflip(mask)
        if random.random() < 0.5:
            image = self.color_jitter(image)

        image = TF.to_tensor(image)
        image = TF.normalize(image, IMAGENET_MEAN, IMAGENET_STD)
        mask = TF.to_tensor(mask)
        mask = (mask > 0.5).float()
        return image, mask


class JointEvalTransform:
    def __init__(self, image_size=256):
        self.image_size = int(image_size)

    def __call__(self, image, mask=None):
        image = image.convert("RGB")
        image = TF.resize(image, (self.image_size, self.image_size), InterpolationMode.BILINEAR)
        image = TF.to_tensor(image)
        image = TF.normalize(image, IMAGENET_MEAN, IMAGENET_STD)
        if mask is None:
            return image
        mask = mask.convert("L")
        mask = TF.resize(mask, (self.image_size, self.image_size), InterpolationMode.NEAREST)
        mask = TF.to_tensor(mask)
        mask = (mask > 0.5).float()
        return image, mask


train_transform = JointTrainTransform(IMAGE_SIZE)
eval_transform = JointEvalTransform(IMAGE_SIZE)


class ISICSegmentationDataset(Dataset):
    def __init__(self, annotations, transform, return_mask=True):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.transform = transform
        self.return_mask = bool(return_mask)

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        image = Image.open(row["image_path"]).convert("RGB")
        if self.return_mask:
            mask = Image.open(row["mask_path"]).convert("L")
            image, mask = self.transform(image, mask)
        else:
            image = self.transform(image, None)
            mask = torch.empty(0)
        return {
            "image": image,
            "mask": mask,
            "fname": row["fname"],
            "image_id": row["image_id"],
            "row_id": int(row["row_id"]),
        }


sample = ISICSegmentationDataset(train_df.head(2), train_transform)[0]
print("Sample image:", sample["image"].shape, sample["image"].dtype)
print("Sample mask:", sample["mask"].shape, sample["mask"].min().item(), sample["mask"].max().item())


## Florence-Prefix with the Same ISIC Screening Policy

Every image receives the same Florence task token `<MORE_DETAILED_CAPTION>`, greedy decoding (`num_beams=1`,
`do_sample=False`, max48 new tokens), without references, filenames, diagnoses or masks.
Remove image-introduction boilerplate and retain the first 5-8 visual words (at most eight; do not fabricate missing words).
No teacher, LoRA, multi-candidate reranking or CLIP image-text ranking is used.

This transfers the DTD **Prefix procedure**, not the DTD 47-class rejection policy. Keep `isic_visible_attributes_v1`
from the Qwen control: reject too-short, diagnosis-containing or token-limit responses; preserve source_long_caption
and natural validity masks. Clinical terms are checked across the full response, not just the retained prefix.
Invalid text has zero CLIP vectors and zero effective gates; its image remains in the dataset and candidate pool.
These are heuristic checks, not proof of factual accuracy or absence of shortcuts. Inspect caption_coverage.csv.

Per-image atomic caches resume interrupted generation. `MAX_NEW_CAPTION_IMAGES=8` is an optional smoke test;
training is blocked until it is reset to 0 and all 2,750 images are exported. A short/truncated output is not silently repaired.


In [ ]:
from __future__ import annotations
import csv
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import re
import shutil
import struct
import subprocess
import sys
import time
from pathlib import Path


def atomic_csv(path, rows, columns):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)

def checkpoint_asset_identity(model_dir):
    model_dir = Path(model_dir)
    files = sorted(path for path in model_dir.iterdir() if path.is_file() and
                   (path.suffix in {".py", ".json", ".safetensors", ".bin", ".txt"}))
    if not any(path.suffix in {".safetensors", ".bin"} for path in files):
        raise ValueError("Checkpoint has no weight files")
    return {path.name: file_sha256(path) for path in files}

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True,
                                    allow_nan=False).encode()).hexdigest()

def read_csv_rows(path, required=()):
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames):
            raise ValueError(f"Missing/duplicate CSV header: {path}")
        if not set(required).issubset(reader.fieldnames):
            raise ValueError(f"Missing required CSV columns: {sorted(set(required) - set(reader.fieldnames))}")
        rows = list(reader)
    if any(None in row or any(value is None for value in row.values()) for row in rows):
        raise ValueError(f"Malformed/incomplete CSV row: {path}; original file was not modified")
    return rows

def clean_dtd_caption(text):
    import re
    text = str(text).strip()
    for _ in range(3):
        before = text
        text = re.sub(r"^(?:(?:the|this)\s+)?(?:image|picture|photo(?:graph)?)\s+"
                      r"(?:is|shows|depicts|features|contains)\s+", "", text, flags=re.I)
        text = re.sub(r"^(?:(?:a|an|the)\s+)?close[ -]up(?:\s+view)?\s+(?:of|showing)\s+",
                      "", text, flags=re.I)
        text = re.sub(r"^(?:a|an|the)(?:\s+|$)", "", text, flags=re.I)
        if text == before:
            break
    words = re.findall(r"[A-Za-z0-9]+(?:['\u2019][A-Za-z0-9]+)?", text)
    selected = []
    normalized_count = 0
    for word in words:
        count = len(re.findall(r"[A-Za-z0-9]+", word))
        if normalized_count + count > 8:
            break
        selected.append(word)
        normalized_count += count
    return " ".join(selected)

def available_rows(directory, records, identity):
    return [row for record in records if (row := cached_row(directory, record, identity)) is not None]

def cache_path(directory, image_id):
    return Path(directory) / (object_hash(image_id) + ".json")

def cached_row(directory, record, identity):
    path = cache_path(directory, record["image_id"])
    if not path.is_file():
        return None
    result = json.loads(path.read_text(encoding="utf-8"))
    if (result.get("identity") != identity or result.get("row", {}).get("image_id") != record["image_id"]
            or result["row"].get("split") != record["split"]):
        raise ValueError(f"Wrong cached image/protocol: {path}")
    return result["row"]

def local_model_configs(roots):
    seen = set()
    for root in roots:
        root = Path(root).expanduser()
        if not root.is_dir():
            continue
        for current, dirs, files in os.walk(root):
            dirs[:] = sorted(name for name in dirs if name not in {
                "images", ".git", "__pycache__", "node_modules", ".cache"})
            if len(Path(current).relative_to(root).parts) >= 9:
                dirs[:] = []
            if "config.json" not in files:
                continue
            path = Path(current).resolve()
            if str(path) not in seen:
                seen.add(str(path))
                yield path

def lock_protocol(path, protocol):
    path = Path(path)
    if path.is_file():
        if json.loads(path.read_text(encoding="utf-8")) != protocol:
            raise ValueError(f"Changed images/model/prompt/runtime/settings. Use a new output directory: {path.parent}")
    else:
        if path.parent.exists() and any(path.parent.iterdir()):
            raise ValueError(f"Refusing to adopt cached output without its protocol: {path.parent}")
        atomic_json(path, protocol)
    return object_hash(protocol)

def qwen_asset_identity(path):
    result = checkpoint_asset_identity(path)
    for template in sorted(Path(path).glob("*.jinja")):
        result[template.name] = file_sha256(template)
    return result

def resolve_clip_model(reference, roots):
    def is_clip(path):
        try:
            return json.loads((path / "config.json").read_text(encoding="utf-8")).get("model_type") == "clip"
        except (OSError, ValueError):
            return False
    if str(reference).strip() and Path(reference).is_dir():
        path = Path(reference).resolve()
        if not is_clip(path):
            raise ValueError(f"Not a full local CLIP checkpoint: {path}")
        return path
    matches = [path for path in local_model_configs(roots) if is_clip(path)]
    if len(matches) != 1:
        raise FileNotFoundError("Set CLIP_MODEL_LOCAL_PATH to the exact full CLIP ViT-B/32 folder")
    return matches[0]

def run_worker(worker_path, payload_path, phase, env, log_path):
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    with Path(log_path).open("a", encoding="utf-8") as log:
        process = subprocess.Popen([sys.executable, str(worker_path), str(payload_path), phase], env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
        finally:
            process.stdout.close()
    if code:
        raise RuntimeError(f"{phase} worker exited with code {code}. See {log_path}; committed image caches are preserved")

def runtime_versions():
    result = {}
    for name in ("torch", "torchvision", "transformers", "accelerate", "tokenizers", "Pillow", "numpy"):
        result[name] = importlib.metadata.version(name)
    return result

def safetensors_parameter_count(files):
    # Read headers only, not tensor contents; verify the attached model's actual scale.
    count = 0
    for path in files:
        with Path(path).open("rb") as handle:
            raw_length = handle.read(8)
            if len(raw_length) != 8:
                raise ValueError(f"Truncated safetensors header: {path}")
            length = struct.unpack("<Q", raw_length)[0]
            if not 2 <= length <= 64 * 1024 * 1024 or length + 8 > Path(path).stat().st_size:
                raise ValueError(f"Invalid safetensors header length: {path}")
            header = json.loads(handle.read(length))
        for name, tensor in header.items():
            if name == "__metadata__":
                continue
            if tensor.get("dtype") not in {"F16", "BF16", "F32"}:
                raise ValueError("Use original floating-point weights, not quantized/FP8 weights, for this comparison")
            shape = tensor.get("shape", [])
            if not all(type(size) is int and size >= 0 for size in shape):
                raise ValueError(f"Invalid tensor shape: {path}: {name}")
            count += math.prod(shape)
    return count

def weight_files(path):
    path = Path(path).resolve()
    index = path / "model.safetensors.index.json"
    if index.is_file():
        names = set(json.loads(index.read_text(encoding="utf-8")).get("weight_map", {}).values())
        if not names:
            raise ValueError(f"Empty safetensors weight index: {index}")
        if any(not isinstance(name, str) or "/" in name or "\\" in name or Path(name).is_absolute() for name in names):
            raise ValueError(f"Weight index must reference shard filenames inside its model folder: {index}")
        files = [path / name for name in sorted(names)]
    else:
        files = [path / "model.safetensors"]
    for file in files:
        # HF snapshot files may be read-only symlinks into a sibling blobs directory.
        if not file.is_file() or file.suffix != ".safetensors":
            raise FileNotFoundError(f"Missing/unsafe local safetensors shard: {file}")
    return files

POLICY = "isic_visible_attributes_v1"

CLINICAL_PATTERN = r"\b(?:melanoma|melanomas|melanocytic|nevus|nevi|naevus|naevi|benign|malignant|malignancy|cancer|cancerous|carcinoma|keratosis|keratoses|diagnosis|diagnosed)\b"

COLUMNS = ["image_id", "split", "raw_caption", "final_caption", "caption_valid", "leakage_detected",
           "clinical_term_detected", "word_count", "source_long_caption", "generation_token_limit_reached",
           "vlm_model", "prompt_version", "caption_input_policy", "captioner_fingerprint", "retry_count"]

def caption_policy(response, token_limit=False):
    caption = clean_dtd_caption(response)
    count = len(re.findall(r"[A-Za-z0-9]+", caption))
    clinical = bool(re.search(CLINICAL_PATTERN, str(response), re.I))
    valid = 5 <= count <= 8 and not clinical and not token_limit
    return caption, valid, clinical, count

def validate_caption_records(records):
    if not records or len({row["image_id"] for row in records}) != len(records):
        raise ValueError("Caption image IDs must be nonempty and unique across splits")
    for row in records:
        if set(row) != {"image_id", "image_path", "image_sha256", "split"}:
            raise ValueError("Caption worker accepts pixels/identity only, never masks, class labels or clinical metadata")
        if row["split"] not in {"train", "val", "test"}:
            raise ValueError("Unexpected split")
        if file_sha256(row["image_path"]) != row["image_sha256"]:
            raise ValueError(f"Image changed: {row['image_id']}")

def validate_text_cache(cache, rows, csv_sha256, identity):
    import torch
    expected_valid = torch.tensor([row["caption_valid"].lower() == "true" for row in rows], dtype=torch.bool)
    vectors, valid = cache["embeddings"], cache["valid"]
    if (cache["identity"] != identity or cache["image_ids"] != [row["image_id"] for row in rows]
            or cache["caption_csv_sha256"] != csv_sha256 or vectors.ndim != 2 or len(vectors) != len(rows)
            or valid.dtype != torch.bool or not torch.equal(valid.cpu(), expected_valid)):
        raise ValueError("Wrong CLIP text cache identity/IDs/shape/masks")
    if not torch.isfinite(vectors).all() or torch.count_nonzero(vectors[~valid]):
        raise ValueError("Invalid captions must have finite zero embeddings")
    if valid.any() and not torch.allclose(vectors[valid].float().norm(dim=1), torch.ones(int(valid.sum())), atol=1e-4):
        raise ValueError("Valid CLIP caption vectors must have unit norm")

METHOD = "florence_prefix_control"

MODEL_ID = "microsoft/Florence-2-base"

TASK_PROMPT = "<MORE_DETAILED_CAPTION>"

def validate_florence_prefix_folder(path):
    path = Path(path).expanduser().resolve()
    config = json.loads((path / "config.json").read_text(encoding="utf-8"))
    if (config.get("model_type") != "florence2"
            or "AutoModelForCausalLM" not in config.get("auto_map", {})):
        raise ValueError("Florence-Prefix requires the original legacy Microsoft Florence-2-base checkpoint")
    required = ("configuration_florence2.py", "modeling_florence2.py", "processing_florence2.py",
                "preprocessor_config.json", "tokenizer_config.json")
    if any(not (path / name).is_file() for name in required):
        raise FileNotFoundError("Incomplete local Florence model code/processor/tokenizer files")
    if not ((path / "tokenizer.json").is_file() or all((path / name).is_file() for name in ("vocab.json", "merges.txt"))):
        raise FileNotFoundError("Missing Florence tokenizer vocabulary")
    if (path / "adapter_config.json").exists() or "-ft" in path.name.lower():
        raise ValueError("Use the original frozen base model, not a LoRA or -ft captioner")
    count = safetensors_parameter_count(weight_files(path))
    if not 180_000_000 <= count <= 300_000_000:
        raise ValueError(f"Expected Florence-2-base scale, not large/quantized/adapter weights: {count:,}")
    return path, count

class IsicFlorencePrefixCaptioner:
    def __init__(self, path, settings):
        import torch
        import transformers
        from transformers import AutoModelForCausalLM, AutoProcessor

        if not transformers.__version__.startswith("4.49."):
            raise RuntimeError("Run the legacy Florence worker with the isolated offline Transformers 4.49 environment")
        self.device = torch.device("cuda:0")
        self.settings = settings
        self.dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        self.processor = AutoProcessor.from_pretrained(str(path), trust_remote_code=True, local_files_only=True)
        model, info = AutoModelForCausalLM.from_pretrained(
            str(path), torch_dtype=self.dtype, trust_remote_code=True, local_files_only=True,
            attn_implementation="eager", low_cpu_mem_usage=True, output_loading_info=True,
        )
        missing = [key for key in info.get("missing_keys", []) if not key.endswith("position_ids")]
        if missing or info.get("mismatched_keys"):
            raise ValueError(f"Incomplete Florence weights: {missing}")
        self.model = model.to(self.device).eval().requires_grad_(False)

    def generate(self, images):
        import torch

        inputs = self.processor(text=[TASK_PROMPT] * len(images), images=images,
                                return_tensors="pt", padding=True)
        moved = {key: value.to(device=self.device, dtype=self.dtype if key == "pixel_values" else value.dtype)
                 for key, value in inputs.items() if torch.is_tensor(value)}
        with torch.inference_mode():
            output = self.model.generate(**moved, max_new_tokens=self.settings["max_new_tokens"],
                                         num_beams=1, do_sample=False, use_cache=True)
        texts = self.processor.batch_decode(output, skip_special_tokens=False)
        if len(texts) != len(images):
            raise ValueError("Florence response count mismatch")
        eos = self.model.generation_config.eos_token_id
        eos = set(eos if isinstance(eos, (list, tuple)) else [eos])
        responses = []
        for text, image, sequence in zip(texts, images, output.cpu().tolist()):
            parsed = self.processor.post_process_generation(text, task=TASK_PROMPT,
                                                           image_size=(image.width, image.height))
            if not isinstance(parsed, dict) or not isinstance(parsed.get(TASK_PROMPT), str):
                raise ValueError("Florence did not return the requested detailed-caption field")
            # Florence is encoder-decoder: its first decoder token can itself be EOS.
            new_tokens = sequence[1:]
            limited = (len(new_tokens) >= self.settings["max_new_tokens"]
                       and not any(token in eos for token in new_tokens))
            responses.append((parsed[TASK_PROMPT].strip(), limited))
        return responses

MODEL_IDS = {METHOD: MODEL_ID}
CAPTION_PROMPT = TASK_PROMPT

def make_generation_row(record, response, token_limit, identity):
    caption, valid, clinical, count = caption_policy(response, token_limit)
    return {"image_id": record["image_id"], "split": record["split"], "raw_caption": caption,
            "final_caption": caption, "caption_valid": valid, "leakage_detected": clinical,
            "clinical_term_detected": clinical, "word_count": count, "source_long_caption": response,
            "generation_token_limit_reached": token_limit, "vlm_model": MODEL_IDS[METHOD],
            "prompt_version": POLICY, "caption_input_policy": POLICY,
            "captioner_fingerprint": identity, "retry_count": 0}

def generate_isic_records(payload, identity, factory=IsicFlorencePrefixCaptioner):
    import torch
    from PIL import Image, ImageOps
    from tqdm.auto import tqdm
    records, settings = payload["records"], payload["settings"]
    directory = Path(payload["output"])
    cache = directory / "generated"
    todo = [row for row in records if cached_row(cache, row, identity) is None]
    if payload.get("max_new_images", 0):
        todo = todo[:payload["max_new_images"]]
    print(f"ISIC captions: cached={len(records) - sum(cached_row(cache, r, identity) is None for r in records)}, queued={len(todo)}", flush=True)
    if todo:
        captioner = factory(payload["model_path"], settings)
        offset, batch_size = 0, settings["batch_size"]
        progress = tqdm(total=len(todo), desc="ISIC Florence-Prefix", unit="image")
        try:
            while offset < len(todo):
                group = todo[offset:offset + batch_size]
                images = []
                try:
                    for record in group:
                        if file_sha256(record["image_path"]) != record["image_sha256"]:
                            raise ValueError("Image changed after hashing")
                        with Image.open(record["image_path"]) as source:
                            images.append(ImageOps.exif_transpose(source).convert("RGB"))
                    responses = captioner.generate(images)
                    if len(responses) != len(group):
                        raise ValueError("Caption batch is incomplete")
                except torch.cuda.OutOfMemoryError:
                    if batch_size == 1:
                        raise RuntimeError("Florence OOM at batch 1; free GPU memory or use a larger GPU. Committed captions are resumable") from None
                    batch_size = max(1, batch_size // 2)
                    gc.collect()
                    torch.cuda.empty_cache()
                    continue
                finally:
                    for image in images:
                        image.close()
                for record, (response, limited) in zip(group, responses):
                    row = make_generation_row(record, response, limited, identity)
                    atomic_json(cache_path(cache, record["image_id"]), {"identity": identity, "row": row})
                offset += len(group)
                progress.update(len(group))
        finally:
            progress.close()
            del captioner
            gc.collect()
            torch.cuda.empty_cache()
    rows = available_rows(cache, records, identity)
    complete = len(rows) == len(records)
    path = directory / ("captions.csv" if complete else "captions_partial.csv")
    atomic_csv(path, rows, COLUMNS)
    status = {"complete": complete, "rows": len(rows), "expected": len(records), "identity": identity,
              "csv_path": str(path.resolve()), "csv_sha256": file_sha256(path),
              "valid_by_split": {role: sum(bool(row["caption_valid"]) for row in rows if row["split"] == role)
                                 for role in ("train", "val", "test")}}
    atomic_json(directory / "status.json", status)
    print(status, flush=True)
    return status

def validated_export(payload):
    directory = Path(payload["output"])
    status = json.loads((directory / "status.json").read_text(encoding="utf-8"))
    if not status["complete"]:
        raise RuntimeError("Caption smoke test/partial output cannot train segmentation; set MAX_NEW_CAPTION_IMAGES=0 and resume")
    path = directory / "captions.csv"
    if file_sha256(path) != status["csv_sha256"]:
        raise ValueError("Caption CSV changed after generation")
    rows = read_csv_rows(path, COLUMNS)
    expected = [(r["image_id"], r["split"]) for r in payload["records"]]
    if [(r["image_id"], r["split"]) for r in rows] != expected:
        raise ValueError("Wrong/missing/duplicate image IDs or split order in captions.csv")
    for row in rows:
        clean, valid, clinical, count = caption_policy(row["source_long_caption"], str(row["generation_token_limit_reached"]).lower() == "true")
        if row["final_caption"] != clean or row["caption_valid"].lower() != str(valid).lower() or int(row["word_count"]) != count:
            raise ValueError("Caption policy fields do not match their original Florence response")
        if row["captioner_fingerprint"] != status["identity"] or row["caption_input_policy"] != POLICY or row["vlm_model"] != MODEL_IDS[METHOD]:
            raise ValueError("Wrong caption generation provenance")
    if not any(row["split"] == "train" and row["caption_valid"].lower() == "true" for row in rows):
        raise ValueError("No usable training captions; inspect CSV and policy, do not force masks/gates on")
    return rows, status

def encode_isic_texts(payload):
    import torch
    from transformers import AutoTokenizer, CLIPTextModelWithProjection
    rows, status = validated_export(payload)
    output = Path(payload["output"])
    protocol = {"schema": 1, "caption_csv_sha256": status["csv_sha256"],
                "clip_assets": qwen_asset_identity(payload["clip_path"]), "policy": POLICY,
                "runtime": runtime_versions(), "code_sha256": payload["code_sha256"]}
    identity = lock_protocol(output / "text_cache" / "protocol.json", protocol)
    target = output / "text_cache" / "clip_text.pt"
    if target.is_file():
        cached = torch.load(target, map_location="cpu", weights_only=True)
        validate_text_cache(cached, rows, status["csv_sha256"], identity)
        atomic_json(output / "text_cache" / "status.json", {"identity": identity, "sha256": file_sha256(target),
                    "valid": int(cached["valid"].sum()), "total": len(rows), "shape": list(cached["embeddings"].shape)})
        print("Reusing CLIP text cache:", target, flush=True)
        return
    tokenizer = AutoTokenizer.from_pretrained(payload["clip_path"], local_files_only=True)
    model, info = CLIPTextModelWithProjection.from_pretrained(payload["clip_path"], local_files_only=True,
                                                            output_loading_info=True)
    missing = [key for key in info.get("missing_keys", []) if not key.endswith("position_ids")]
    if missing or info.get("mismatched_keys"):
        raise ValueError(f"Incomplete CLIP text weights: {missing}")
    model = model.to("cuda:0").eval().requires_grad_(False)
    valid = torch.tensor([row["caption_valid"].lower() == "true" for row in rows], dtype=torch.bool)
    vectors = torch.zeros((len(rows), model.config.projection_dim), dtype=torch.float32)
    positions = valid.nonzero().flatten().tolist()
    for start in range(0, len(positions), 128):
        group = positions[start:start + 128]
        tokens = tokenizer([rows[i]["final_caption"] for i in group], padding=True, truncation=True,
                           max_length=model.config.max_position_embeddings, return_tensors="pt")
        tokens = {key: value.to("cuda:0") for key, value in tokens.items()}
        with torch.inference_mode():
            values = torch.nn.functional.normalize(model(**tokens).text_embeds.float(), dim=1)
        vectors[group] = values.cpu()
    temporary = target.with_suffix(".pt.tmp")
    torch.save({"identity": identity, "image_ids": [row["image_id"] for row in rows],
                "embeddings": vectors, "valid": valid, "caption_csv_sha256": status["csv_sha256"]}, temporary)
    temporary.replace(target)
    atomic_json(output / "text_cache" / "status.json", {"identity": identity, "sha256": file_sha256(target),
                "valid": int(valid.sum()), "total": len(rows), "shape": list(vectors.shape)})
    print("CLIP vectors:", tuple(vectors.shape), "valid:", int(valid.sum()), flush=True)

def florence_checkpoint(model_ref, roots):
    def checked(path):
        path = Path(path)
        try:
            config = json.loads((path / "config.json").read_text(encoding="utf-8"))
        except (OSError, ValueError):
            return None
        if config.get("model_type") != "florence2":
            return None
        indexes = list(path.glob("*.index.json"))
        weights = any((path / name).is_file() for name in ("model.safetensors", "pytorch_model.bin"))
        for index in indexes:
            try:
                shards = set(json.loads(index.read_text(encoding="utf-8")).get("weight_map", {}).values())
                weights |= bool(shards) and all((path / shard).is_file() for shard in shards)
            except (OSError, ValueError):
                continue
        if not weights:
            return None
        backend = "legacy" if "AutoModelForCausalLM" in config.get("auto_map", {}) else "native"
        if backend == "legacy" and not all((path / name).is_file() for name in (
                "configuration_florence2.py", "modeling_florence2.py", "processing_florence2.py")):
            return None
        return path.resolve(), backend

    requested = Path(model_ref).expanduser()
    if requested.is_dir():
        result = checked(requested)
        if result:
            return result
        for config in sorted(requested.rglob("config.json")):
            result = checked(config.parent)
            if result:
                return result
        raise FileNotFoundError(f"Florence folder is incomplete: {requested}; need config, weights, processor/tokenizer and local model code.")
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for current, dirs, files in os.walk(root):
            dirs[:] = sorted(name for name in dirs if name not in {"images", "__pycache__", ".git"})
            if len(Path(current).relative_to(root).parts) >= 8:
                dirs[:] = []
            if "config.json" in files:
                result = checked(current)
                if result:
                    return result
    raise FileNotFoundError(
        f"No complete local Florence-2 checkpoint found for {model_ref}. "
        "Attach notebook-mt-moi/Florence-2-base or set FLORENCE_MODEL_REF to the correct local folder. No download is attempted."
    )

def offline_caption_wheels(roots, destination):
    import zipfile
    destination = Path(destination)
    wheel_dirs = set()
    prefixes = ("transformers-", "tokenizers-", "huggingface_hub-", "accelerate-", "einops-", "timm-")
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for current, dirs, files in os.walk(root):
            dirs[:] = sorted(name for name in dirs if name not in {"images", "__pycache__", ".git", "Florence-2-base"})
            if len(Path(current).relative_to(root).parts) >= 7:
                dirs[:] = []
            for name in sorted(files):
                path = Path(current) / name
                if name.endswith(".whl"):
                    wheel_dirs.add(str(path.parent))
                elif name.lower() in {"offline_wheels.zip", "wheels.zip"}:
                    with zipfile.ZipFile(path) as archive:
                        for member in archive.infolist():
                            filename = Path(member.filename.replace("\\", "/")).name
                            if not filename.endswith(".whl") or not filename.startswith(prefixes):
                                continue
                            destination.mkdir(parents=True, exist_ok=True)
                            target = destination / filename
                            if not target.is_file() or target.stat().st_size != member.file_size:
                                with archive.open(member) as source, target.open("wb") as handle:
                                    shutil.copyfileobj(source, handle)
                            wheel_dirs.add(str(destination))
    return sorted(wheel_dirs)

def caption_runtime_environment(backend, asset_root, runtime_dir):
    import subprocess
    import sys
    runtime_dir = Path(runtime_dir)
    runtime_dir.mkdir(parents=True, exist_ok=True)
    env = dict(os.environ)
    env.update(TOKENIZERS_PARALLELISM="false", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
               HF_MODULES_CACHE=str(runtime_dir / "hf_modules"), PYTHONUNBUFFERED="1")
    target = runtime_dir / "legacy_packages"
    probe = (
        "import transformers; "
        + ("assert transformers.__version__.startswith('4.49.'), transformers.__version__; " if backend == "legacy" else "")
        + "from transformers import AutoProcessor, AutoTokenizer, CLIPModel; "
        + ("from transformers import AutoModelForCausalLM; " if backend == "legacy" else
           "from transformers import Florence2ForConditionalGeneration; ")
        + "import einops, timm, accelerate; print(transformers.__version__)"
    )
    if target.is_dir():
        env["PYTHONPATH"] = str(target) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    result = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)
    if result.returncode == 0:
        print("Caption subprocess Transformers:", result.stdout.strip())
        return env
    if backend != "legacy":
        raise RuntimeError("The local native Florence checkpoint cannot load in this runtime. "
                           "Use a matching native Transformers installation or the attached legacy model + offline 4.49 wheels.\n"
                           + result.stderr[-3000:])
    wheel_dirs = offline_caption_wheels([Path(asset_root), Path("/kaggle/input")], runtime_dir / "wheels")
    if not wheel_dirs:
        raise FileNotFoundError(
            "The attached Microsoft Florence checkpoint needs Transformers 4.49.x. "
            f"No offline wheels found under {asset_root}. Attach offline_wheels.zip/wheels.zip or a wheel folder "
            "with transformers==4.49.0, tokenizers 0.21.x and huggingface-hub <1 compatible with this Python version. "
            "The notebook never enables Internet or modifies the kernel's Transformers."
        )
    command = [sys.executable, "-m", "pip", "install", "--no-index", "--only-binary=:all:",
               "--no-deps", "--upgrade", "--target", str(target), "transformers==4.49.0",
               "tokenizers>=0.21,<0.22", "huggingface-hub>=0.26,<1"]
    import importlib.util
    for module in ("einops", "timm", "accelerate"):
        if importlib.util.find_spec(module) is None:
            command.append(module)
    for directory in wheel_dirs:
        command.extend(["--find-links", directory])
    print("Preparing isolated Florence dependencies from attached offline wheels; the notebook kernel is unchanged.")
    installed = subprocess.run(command, env=env, capture_output=True, text=True)
    if installed.returncode:
        raise RuntimeError("Offline Florence dependency setup failed. Wheels must match Kaggle's Python/platform.\n"
                           + installed.stdout[-2000:] + installed.stderr[-3000:])
    base_pythonpath = os.environ.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(target) + (os.pathsep + base_pythonpath if base_pythonpath else "")
    checked = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)
    if checked.returncode:
        raise RuntimeError("The offline Florence environment still lacks a compatible dependency. "
                           "No kernel packages were changed.\n" + checked.stderr[-4000:])
    print("Isolated caption subprocess Transformers:", checked.stdout.strip())
    return env

def florence_worker_main(request_path, phase, *, generate_records, encode_texts):
    import torch

    payload = json.loads(Path(request_path).read_text(encoding="utf-8"))
    validate_caption_records(payload["records"])
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a CUDA GPU")
    settings = payload["settings"]
    if type(payload.get("max_new_images", 0)) is not int or payload.get("max_new_images", 0) < 0:
        raise ValueError("max_new_images must be a nonnegative integer")
    for key in ("batch_size", "max_new_tokens"):
        if type(settings[key]) is not int or settings[key] < 1:
            raise ValueError(f"{key} must be a positive integer")
    if type(settings["seed"]) is not int or not 0 <= settings["seed"] < 2**32:
        raise ValueError("Caption seed must be an integer in [0, 2**32)")
    torch.manual_seed(settings["seed"])
    torch.cuda.manual_seed_all(settings["seed"])
    if phase == "generate":
        from importlib.metadata import version
        if not version("transformers").startswith("4.49."):
            raise RuntimeError("Legacy Florence generation must use the isolated Transformers 4.49 runtime")
        path, count = validate_florence_prefix_folder(payload["model_path"])
        protocol = {"schema": 1, "method": METHOD, "model_id": MODEL_ID,
                    "model_assets": checkpoint_asset_identity(path), "parameters": count,
                    "prompt": TASK_PROMPT, "decoding": "greedy_beam1_prefix_cleanup_first8",
                    "dtype": "bfloat16" if torch.cuda.is_bf16_supported() else "float16",
                    "policy": "isic_visible_attributes_v1",
                    "images": [{key: row[key] for key in ("image_id", "split", "image_sha256")}
                               for row in payload["records"]],
                    "settings": settings, "runtime": runtime_versions(), "code_sha256": payload["code_sha256"]}
        output = Path(payload["output"])
        identity = lock_protocol(output / "generation_protocol" / "protocol.json", protocol)
        if not (output / "status.json").exists() and (output / "captions.csv").exists():
            raise ValueError("Unregistered captions.csv; use a fresh output root")
        generate_records(payload, identity)
    elif phase == "encode":
        encode_texts(payload)
    else:
        raise ValueError("Phase must be generate or encode")


In [ ]:
ISIC_WORKER_SOURCE = 'from __future__ import annotations\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nimport re\nimport shutil\nimport struct\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\n\ndef atomic_csv(path, rows, columns):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n        handle.flush()\n        os.fsync(handle.fileno())\n    temporary.replace(path)\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\ndef checkpoint_asset_identity(model_dir):\n    model_dir = Path(model_dir)\n    files = sorted(path for path in model_dir.iterdir() if path.is_file() and\n                   (path.suffix in {".py", ".json", ".safetensors", ".bin", ".txt"}))\n    if not any(path.suffix in {".safetensors", ".bin"} for path in files):\n        raise ValueError("Checkpoint has no weight files")\n    return {path.name: file_sha256(path) for path in files}\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\ndef object_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True,\n                                    allow_nan=False).encode()).hexdigest()\n\ndef read_csv_rows(path, required=()):\n    with Path(path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames):\n            raise ValueError(f"Missing/duplicate CSV header: {path}")\n        if not set(required).issubset(reader.fieldnames):\n            raise ValueError(f"Missing required CSV columns: {sorted(set(required) - set(reader.fieldnames))}")\n        rows = list(reader)\n    if any(None in row or any(value is None for value in row.values()) for row in rows):\n        raise ValueError(f"Malformed/incomplete CSV row: {path}; original file was not modified")\n    return rows\n\ndef clean_dtd_caption(text):\n    import re\n    text = str(text).strip()\n    for _ in range(3):\n        before = text\n        text = re.sub(r"^(?:(?:the|this)\\s+)?(?:image|picture|photo(?:graph)?)\\s+"\n                      r"(?:is|shows|depicts|features|contains)\\s+", "", text, flags=re.I)\n        text = re.sub(r"^(?:(?:a|an|the)\\s+)?close[ -]up(?:\\s+view)?\\s+(?:of|showing)\\s+",\n                      "", text, flags=re.I)\n        text = re.sub(r"^(?:a|an|the)(?:\\s+|$)", "", text, flags=re.I)\n        if text == before:\n            break\n    words = re.findall(r"[A-Za-z0-9]+(?:[\'\\u2019][A-Za-z0-9]+)?", text)\n    selected = []\n    normalized_count = 0\n    for word in words:\n        count = len(re.findall(r"[A-Za-z0-9]+", word))\n        if normalized_count + count > 8:\n            break\n        selected.append(word)\n        normalized_count += count\n    return " ".join(selected)\n\ndef available_rows(directory, records, identity):\n    return [row for record in records if (row := cached_row(directory, record, identity)) is not None]\n\ndef cache_path(directory, image_id):\n    return Path(directory) / (object_hash(image_id) + ".json")\n\ndef cached_row(directory, record, identity):\n    path = cache_path(directory, record["image_id"])\n    if not path.is_file():\n        return None\n    result = json.loads(path.read_text(encoding="utf-8"))\n    if (result.get("identity") != identity or result.get("row", {}).get("image_id") != record["image_id"]\n            or result["row"].get("split") != record["split"]):\n        raise ValueError(f"Wrong cached image/protocol: {path}")\n    return result["row"]\n\ndef local_model_configs(roots):\n    seen = set()\n    for root in roots:\n        root = Path(root).expanduser()\n        if not root.is_dir():\n            continue\n        for current, dirs, files in os.walk(root):\n            dirs[:] = sorted(name for name in dirs if name not in {\n                "images", ".git", "__pycache__", "node_modules", ".cache"})\n            if len(Path(current).relative_to(root).parts) >= 9:\n                dirs[:] = []\n            if "config.json" not in files:\n                continue\n            path = Path(current).resolve()\n            if str(path) not in seen:\n                seen.add(str(path))\n                yield path\n\ndef lock_protocol(path, protocol):\n    path = Path(path)\n    if path.is_file():\n        if json.loads(path.read_text(encoding="utf-8")) != protocol:\n            raise ValueError(f"Changed images/model/prompt/runtime/settings. Use a new output directory: {path.parent}")\n    else:\n        if path.parent.exists() and any(path.parent.iterdir()):\n            raise ValueError(f"Refusing to adopt cached output without its protocol: {path.parent}")\n        atomic_json(path, protocol)\n    return object_hash(protocol)\n\ndef qwen_asset_identity(path):\n    result = checkpoint_asset_identity(path)\n    for template in sorted(Path(path).glob("*.jinja")):\n        result[template.name] = file_sha256(template)\n    return result\n\ndef resolve_clip_model(reference, roots):\n    def is_clip(path):\n        try:\n            return json.loads((path / "config.json").read_text(encoding="utf-8")).get("model_type") == "clip"\n        except (OSError, ValueError):\n            return False\n    if str(reference).strip() and Path(reference).is_dir():\n        path = Path(reference).resolve()\n        if not is_clip(path):\n            raise ValueError(f"Not a full local CLIP checkpoint: {path}")\n        return path\n    matches = [path for path in local_model_configs(roots) if is_clip(path)]\n    if len(matches) != 1:\n        raise FileNotFoundError("Set CLIP_MODEL_LOCAL_PATH to the exact full CLIP ViT-B/32 folder")\n    return matches[0]\n\ndef run_worker(worker_path, payload_path, phase, env, log_path):\n    Path(log_path).parent.mkdir(parents=True, exist_ok=True)\n    with Path(log_path).open("a", encoding="utf-8") as log:\n        process = subprocess.Popen([sys.executable, str(worker_path), str(payload_path), phase], env=env,\n            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)\n        try:\n            for line in process.stdout:\n                print(line, end="", flush=True)\n                log.write(line)\n                log.flush()\n            code = process.wait()\n        except BaseException:\n            process.terminate()\n            process.wait()\n            raise\n        finally:\n            process.stdout.close()\n    if code:\n        raise RuntimeError(f"{phase} worker exited with code {code}. See {log_path}; committed image caches are preserved")\n\ndef runtime_versions():\n    result = {}\n    for name in ("torch", "torchvision", "transformers", "accelerate", "tokenizers", "Pillow", "numpy"):\n        result[name] = importlib.metadata.version(name)\n    return result\n\ndef safetensors_parameter_count(files):\n    # Read headers only, not tensor contents; verify the attached model\'s actual scale.\n    count = 0\n    for path in files:\n        with Path(path).open("rb") as handle:\n            raw_length = handle.read(8)\n            if len(raw_length) != 8:\n                raise ValueError(f"Truncated safetensors header: {path}")\n            length = struct.unpack("<Q", raw_length)[0]\n            if not 2 <= length <= 64 * 1024 * 1024 or length + 8 > Path(path).stat().st_size:\n                raise ValueError(f"Invalid safetensors header length: {path}")\n            header = json.loads(handle.read(length))\n        for name, tensor in header.items():\n            if name == "__metadata__":\n                continue\n            if tensor.get("dtype") not in {"F16", "BF16", "F32"}:\n                raise ValueError("Use original floating-point weights, not quantized/FP8 weights, for this comparison")\n            shape = tensor.get("shape", [])\n            if not all(type(size) is int and size >= 0 for size in shape):\n                raise ValueError(f"Invalid tensor shape: {path}: {name}")\n            count += math.prod(shape)\n    return count\n\ndef weight_files(path):\n    path = Path(path).resolve()\n    index = path / "model.safetensors.index.json"\n    if index.is_file():\n        names = set(json.loads(index.read_text(encoding="utf-8")).get("weight_map", {}).values())\n        if not names:\n            raise ValueError(f"Empty safetensors weight index: {index}")\n        if any(not isinstance(name, str) or "/" in name or "\\\\" in name or Path(name).is_absolute() for name in names):\n            raise ValueError(f"Weight index must reference shard filenames inside its model folder: {index}")\n        files = [path / name for name in sorted(names)]\n    else:\n        files = [path / "model.safetensors"]\n    for file in files:\n        # HF snapshot files may be read-only symlinks into a sibling blobs directory.\n        if not file.is_file() or file.suffix != ".safetensors":\n            raise FileNotFoundError(f"Missing/unsafe local safetensors shard: {file}")\n    return files\n\nPOLICY = "isic_visible_attributes_v1"\n\nCLINICAL_PATTERN = r"\\b(?:melanoma|melanomas|melanocytic|nevus|nevi|naevus|naevi|benign|malignant|malignancy|cancer|cancerous|carcinoma|keratosis|keratoses|diagnosis|diagnosed)\\b"\n\nCOLUMNS = ["image_id", "split", "raw_caption", "final_caption", "caption_valid", "leakage_detected",\n           "clinical_term_detected", "word_count", "source_long_caption", "generation_token_limit_reached",\n           "vlm_model", "prompt_version", "caption_input_policy", "captioner_fingerprint", "retry_count"]\n\ndef caption_policy(response, token_limit=False):\n    caption = clean_dtd_caption(response)\n    count = len(re.findall(r"[A-Za-z0-9]+", caption))\n    clinical = bool(re.search(CLINICAL_PATTERN, str(response), re.I))\n    valid = 5 <= count <= 8 and not clinical and not token_limit\n    return caption, valid, clinical, count\n\ndef validate_caption_records(records):\n    if not records or len({row["image_id"] for row in records}) != len(records):\n        raise ValueError("Caption image IDs must be nonempty and unique across splits")\n    for row in records:\n        if set(row) != {"image_id", "image_path", "image_sha256", "split"}:\n            raise ValueError("Caption worker accepts pixels/identity only, never masks, class labels or clinical metadata")\n        if row["split"] not in {"train", "val", "test"}:\n            raise ValueError("Unexpected split")\n        if file_sha256(row["image_path"]) != row["image_sha256"]:\n            raise ValueError(f"Image changed: {row[\'image_id\']}")\n\ndef validate_text_cache(cache, rows, csv_sha256, identity):\n    import torch\n    expected_valid = torch.tensor([row["caption_valid"].lower() == "true" for row in rows], dtype=torch.bool)\n    vectors, valid = cache["embeddings"], cache["valid"]\n    if (cache["identity"] != identity or cache["image_ids"] != [row["image_id"] for row in rows]\n            or cache["caption_csv_sha256"] != csv_sha256 or vectors.ndim != 2 or len(vectors) != len(rows)\n            or valid.dtype != torch.bool or not torch.equal(valid.cpu(), expected_valid)):\n        raise ValueError("Wrong CLIP text cache identity/IDs/shape/masks")\n    if not torch.isfinite(vectors).all() or torch.count_nonzero(vectors[~valid]):\n        raise ValueError("Invalid captions must have finite zero embeddings")\n    if valid.any() and not torch.allclose(vectors[valid].float().norm(dim=1), torch.ones(int(valid.sum())), atol=1e-4):\n        raise ValueError("Valid CLIP caption vectors must have unit norm")\n\nMETHOD = "florence_prefix_control"\n\nMODEL_ID = "microsoft/Florence-2-base"\n\nTASK_PROMPT = "<MORE_DETAILED_CAPTION>"\n\ndef validate_florence_prefix_folder(path):\n    path = Path(path).expanduser().resolve()\n    config = json.loads((path / "config.json").read_text(encoding="utf-8"))\n    if (config.get("model_type") != "florence2"\n            or "AutoModelForCausalLM" not in config.get("auto_map", {})):\n        raise ValueError("Florence-Prefix requires the original legacy Microsoft Florence-2-base checkpoint")\n    required = ("configuration_florence2.py", "modeling_florence2.py", "processing_florence2.py",\n                "preprocessor_config.json", "tokenizer_config.json")\n    if any(not (path / name).is_file() for name in required):\n        raise FileNotFoundError("Incomplete local Florence model code/processor/tokenizer files")\n    if not ((path / "tokenizer.json").is_file() or all((path / name).is_file() for name in ("vocab.json", "merges.txt"))):\n        raise FileNotFoundError("Missing Florence tokenizer vocabulary")\n    if (path / "adapter_config.json").exists() or "-ft" in path.name.lower():\n        raise ValueError("Use the original frozen base model, not a LoRA or -ft captioner")\n    count = safetensors_parameter_count(weight_files(path))\n    if not 180_000_000 <= count <= 300_000_000:\n        raise ValueError(f"Expected Florence-2-base scale, not large/quantized/adapter weights: {count:,}")\n    return path, count\n\nclass IsicFlorencePrefixCaptioner:\n    def __init__(self, path, settings):\n        import torch\n        import transformers\n        from transformers import AutoModelForCausalLM, AutoProcessor\n\n        if not transformers.__version__.startswith("4.49."):\n            raise RuntimeError("Run the legacy Florence worker with the isolated offline Transformers 4.49 environment")\n        self.device = torch.device("cuda:0")\n        self.settings = settings\n        self.dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n        self.processor = AutoProcessor.from_pretrained(str(path), trust_remote_code=True, local_files_only=True)\n        model, info = AutoModelForCausalLM.from_pretrained(\n            str(path), torch_dtype=self.dtype, trust_remote_code=True, local_files_only=True,\n            attn_implementation="eager", low_cpu_mem_usage=True, output_loading_info=True,\n        )\n        missing = [key for key in info.get("missing_keys", []) if not key.endswith("position_ids")]\n        if missing or info.get("mismatched_keys"):\n            raise ValueError(f"Incomplete Florence weights: {missing}")\n        self.model = model.to(self.device).eval().requires_grad_(False)\n\n    def generate(self, images):\n        import torch\n\n        inputs = self.processor(text=[TASK_PROMPT] * len(images), images=images,\n                                return_tensors="pt", padding=True)\n        moved = {key: value.to(device=self.device, dtype=self.dtype if key == "pixel_values" else value.dtype)\n                 for key, value in inputs.items() if torch.is_tensor(value)}\n        with torch.inference_mode():\n            output = self.model.generate(**moved, max_new_tokens=self.settings["max_new_tokens"],\n                                         num_beams=1, do_sample=False, use_cache=True)\n        texts = self.processor.batch_decode(output, skip_special_tokens=False)\n        if len(texts) != len(images):\n            raise ValueError("Florence response count mismatch")\n        eos = self.model.generation_config.eos_token_id\n        eos = set(eos if isinstance(eos, (list, tuple)) else [eos])\n        responses = []\n        for text, image, sequence in zip(texts, images, output.cpu().tolist()):\n            parsed = self.processor.post_process_generation(text, task=TASK_PROMPT,\n                                                           image_size=(image.width, image.height))\n            if not isinstance(parsed, dict) or not isinstance(parsed.get(TASK_PROMPT), str):\n                raise ValueError("Florence did not return the requested detailed-caption field")\n            # Florence is encoder-decoder: its first decoder token can itself be EOS.\n            new_tokens = sequence[1:]\n            limited = (len(new_tokens) >= self.settings["max_new_tokens"]\n                       and not any(token in eos for token in new_tokens))\n            responses.append((parsed[TASK_PROMPT].strip(), limited))\n        return responses\n\nMODEL_IDS = {METHOD: MODEL_ID}\nCAPTION_PROMPT = TASK_PROMPT\n\ndef make_generation_row(record, response, token_limit, identity):\n    caption, valid, clinical, count = caption_policy(response, token_limit)\n    return {"image_id": record["image_id"], "split": record["split"], "raw_caption": caption,\n            "final_caption": caption, "caption_valid": valid, "leakage_detected": clinical,\n            "clinical_term_detected": clinical, "word_count": count, "source_long_caption": response,\n            "generation_token_limit_reached": token_limit, "vlm_model": MODEL_IDS[METHOD],\n            "prompt_version": POLICY, "caption_input_policy": POLICY,\n            "captioner_fingerprint": identity, "retry_count": 0}\n\ndef generate_isic_records(payload, identity, factory=IsicFlorencePrefixCaptioner):\n    import torch\n    from PIL import Image, ImageOps\n    from tqdm.auto import tqdm\n    records, settings = payload["records"], payload["settings"]\n    directory = Path(payload["output"])\n    cache = directory / "generated"\n    todo = [row for row in records if cached_row(cache, row, identity) is None]\n    if payload.get("max_new_images", 0):\n        todo = todo[:payload["max_new_images"]]\n    print(f"ISIC captions: cached={len(records) - sum(cached_row(cache, r, identity) is None for r in records)}, queued={len(todo)}", flush=True)\n    if todo:\n        captioner = factory(payload["model_path"], settings)\n        offset, batch_size = 0, settings["batch_size"]\n        progress = tqdm(total=len(todo), desc="ISIC Florence-Prefix", unit="image")\n        try:\n            while offset < len(todo):\n                group = todo[offset:offset + batch_size]\n                images = []\n                try:\n                    for record in group:\n                        if file_sha256(record["image_path"]) != record["image_sha256"]:\n                            raise ValueError("Image changed after hashing")\n                        with Image.open(record["image_path"]) as source:\n                            images.append(ImageOps.exif_transpose(source).convert("RGB"))\n                    responses = captioner.generate(images)\n                    if len(responses) != len(group):\n                        raise ValueError("Caption batch is incomplete")\n                except torch.cuda.OutOfMemoryError:\n                    if batch_size == 1:\n                        raise RuntimeError("Florence OOM at batch 1; free GPU memory or use a larger GPU. Committed captions are resumable") from None\n                    batch_size = max(1, batch_size // 2)\n                    gc.collect()\n                    torch.cuda.empty_cache()\n                    continue\n                finally:\n                    for image in images:\n                        image.close()\n                for record, (response, limited) in zip(group, responses):\n                    row = make_generation_row(record, response, limited, identity)\n                    atomic_json(cache_path(cache, record["image_id"]), {"identity": identity, "row": row})\n                offset += len(group)\n                progress.update(len(group))\n        finally:\n            progress.close()\n            del captioner\n            gc.collect()\n            torch.cuda.empty_cache()\n    rows = available_rows(cache, records, identity)\n    complete = len(rows) == len(records)\n    path = directory / ("captions.csv" if complete else "captions_partial.csv")\n    atomic_csv(path, rows, COLUMNS)\n    status = {"complete": complete, "rows": len(rows), "expected": len(records), "identity": identity,\n              "csv_path": str(path.resolve()), "csv_sha256": file_sha256(path),\n              "valid_by_split": {role: sum(bool(row["caption_valid"]) for row in rows if row["split"] == role)\n                                 for role in ("train", "val", "test")}}\n    atomic_json(directory / "status.json", status)\n    print(status, flush=True)\n    return status\n\ndef validated_export(payload):\n    directory = Path(payload["output"])\n    status = json.loads((directory / "status.json").read_text(encoding="utf-8"))\n    if not status["complete"]:\n        raise RuntimeError("Caption smoke test/partial output cannot train segmentation; set MAX_NEW_CAPTION_IMAGES=0 and resume")\n    path = directory / "captions.csv"\n    if file_sha256(path) != status["csv_sha256"]:\n        raise ValueError("Caption CSV changed after generation")\n    rows = read_csv_rows(path, COLUMNS)\n    expected = [(r["image_id"], r["split"]) for r in payload["records"]]\n    if [(r["image_id"], r["split"]) for r in rows] != expected:\n        raise ValueError("Wrong/missing/duplicate image IDs or split order in captions.csv")\n    for row in rows:\n        clean, valid, clinical, count = caption_policy(row["source_long_caption"], str(row["generation_token_limit_reached"]).lower() == "true")\n        if row["final_caption"] != clean or row["caption_valid"].lower() != str(valid).lower() or int(row["word_count"]) != count:\n            raise ValueError("Caption policy fields do not match their original Florence response")\n        if row["captioner_fingerprint"] != status["identity"] or row["caption_input_policy"] != POLICY or row["vlm_model"] != MODEL_IDS[METHOD]:\n            raise ValueError("Wrong caption generation provenance")\n    if not any(row["split"] == "train" and row["caption_valid"].lower() == "true" for row in rows):\n        raise ValueError("No usable training captions; inspect CSV and policy, do not force masks/gates on")\n    return rows, status\n\ndef encode_isic_texts(payload):\n    import torch\n    from transformers import AutoTokenizer, CLIPTextModelWithProjection\n    rows, status = validated_export(payload)\n    output = Path(payload["output"])\n    protocol = {"schema": 1, "caption_csv_sha256": status["csv_sha256"],\n                "clip_assets": qwen_asset_identity(payload["clip_path"]), "policy": POLICY,\n                "runtime": runtime_versions(), "code_sha256": payload["code_sha256"]}\n    identity = lock_protocol(output / "text_cache" / "protocol.json", protocol)\n    target = output / "text_cache" / "clip_text.pt"\n    if target.is_file():\n        cached = torch.load(target, map_location="cpu", weights_only=True)\n        validate_text_cache(cached, rows, status["csv_sha256"], identity)\n        atomic_json(output / "text_cache" / "status.json", {"identity": identity, "sha256": file_sha256(target),\n                    "valid": int(cached["valid"].sum()), "total": len(rows), "shape": list(cached["embeddings"].shape)})\n        print("Reusing CLIP text cache:", target, flush=True)\n        return\n    tokenizer = AutoTokenizer.from_pretrained(payload["clip_path"], local_files_only=True)\n    model, info = CLIPTextModelWithProjection.from_pretrained(payload["clip_path"], local_files_only=True,\n                                                            output_loading_info=True)\n    missing = [key for key in info.get("missing_keys", []) if not key.endswith("position_ids")]\n    if missing or info.get("mismatched_keys"):\n        raise ValueError(f"Incomplete CLIP text weights: {missing}")\n    model = model.to("cuda:0").eval().requires_grad_(False)\n    valid = torch.tensor([row["caption_valid"].lower() == "true" for row in rows], dtype=torch.bool)\n    vectors = torch.zeros((len(rows), model.config.projection_dim), dtype=torch.float32)\n    positions = valid.nonzero().flatten().tolist()\n    for start in range(0, len(positions), 128):\n        group = positions[start:start + 128]\n        tokens = tokenizer([rows[i]["final_caption"] for i in group], padding=True, truncation=True,\n                           max_length=model.config.max_position_embeddings, return_tensors="pt")\n        tokens = {key: value.to("cuda:0") for key, value in tokens.items()}\n        with torch.inference_mode():\n            values = torch.nn.functional.normalize(model(**tokens).text_embeds.float(), dim=1)\n        vectors[group] = values.cpu()\n    temporary = target.with_suffix(".pt.tmp")\n    torch.save({"identity": identity, "image_ids": [row["image_id"] for row in rows],\n                "embeddings": vectors, "valid": valid, "caption_csv_sha256": status["csv_sha256"]}, temporary)\n    temporary.replace(target)\n    atomic_json(output / "text_cache" / "status.json", {"identity": identity, "sha256": file_sha256(target),\n                "valid": int(valid.sum()), "total": len(rows), "shape": list(vectors.shape)})\n    print("CLIP vectors:", tuple(vectors.shape), "valid:", int(valid.sum()), flush=True)\n\ndef florence_checkpoint(model_ref, roots):\n    def checked(path):\n        path = Path(path)\n        try:\n            config = json.loads((path / "config.json").read_text(encoding="utf-8"))\n        except (OSError, ValueError):\n            return None\n        if config.get("model_type") != "florence2":\n            return None\n        indexes = list(path.glob("*.index.json"))\n        weights = any((path / name).is_file() for name in ("model.safetensors", "pytorch_model.bin"))\n        for index in indexes:\n            try:\n                shards = set(json.loads(index.read_text(encoding="utf-8")).get("weight_map", {}).values())\n                weights |= bool(shards) and all((path / shard).is_file() for shard in shards)\n            except (OSError, ValueError):\n                continue\n        if not weights:\n            return None\n        backend = "legacy" if "AutoModelForCausalLM" in config.get("auto_map", {}) else "native"\n        if backend == "legacy" and not all((path / name).is_file() for name in (\n                "configuration_florence2.py", "modeling_florence2.py", "processing_florence2.py")):\n            return None\n        return path.resolve(), backend\n\n    requested = Path(model_ref).expanduser()\n    if requested.is_dir():\n        result = checked(requested)\n        if result:\n            return result\n        for config in sorted(requested.rglob("config.json")):\n            result = checked(config.parent)\n            if result:\n                return result\n        raise FileNotFoundError(f"Florence folder is incomplete: {requested}; need config, weights, processor/tokenizer and local model code.")\n    for root in roots:\n        root = Path(root)\n        if not root.is_dir():\n            continue\n        for current, dirs, files in os.walk(root):\n            dirs[:] = sorted(name for name in dirs if name not in {"images", "__pycache__", ".git"})\n            if len(Path(current).relative_to(root).parts) >= 8:\n                dirs[:] = []\n            if "config.json" in files:\n                result = checked(current)\n                if result:\n                    return result\n    raise FileNotFoundError(\n        f"No complete local Florence-2 checkpoint found for {model_ref}. "\n        "Attach notebook-mt-moi/Florence-2-base or set FLORENCE_MODEL_REF to the correct local folder. No download is attempted."\n    )\n\ndef offline_caption_wheels(roots, destination):\n    import zipfile\n    destination = Path(destination)\n    wheel_dirs = set()\n    prefixes = ("transformers-", "tokenizers-", "huggingface_hub-", "accelerate-", "einops-", "timm-")\n    for root in roots:\n        root = Path(root)\n        if not root.is_dir():\n            continue\n        for current, dirs, files in os.walk(root):\n            dirs[:] = sorted(name for name in dirs if name not in {"images", "__pycache__", ".git", "Florence-2-base"})\n            if len(Path(current).relative_to(root).parts) >= 7:\n                dirs[:] = []\n            for name in sorted(files):\n                path = Path(current) / name\n                if name.endswith(".whl"):\n                    wheel_dirs.add(str(path.parent))\n                elif name.lower() in {"offline_wheels.zip", "wheels.zip"}:\n                    with zipfile.ZipFile(path) as archive:\n                        for member in archive.infolist():\n                            filename = Path(member.filename.replace("\\\\", "/")).name\n                            if not filename.endswith(".whl") or not filename.startswith(prefixes):\n                                continue\n                            destination.mkdir(parents=True, exist_ok=True)\n                            target = destination / filename\n                            if not target.is_file() or target.stat().st_size != member.file_size:\n                                with archive.open(member) as source, target.open("wb") as handle:\n                                    shutil.copyfileobj(source, handle)\n                            wheel_dirs.add(str(destination))\n    return sorted(wheel_dirs)\n\ndef caption_runtime_environment(backend, asset_root, runtime_dir):\n    import subprocess\n    import sys\n    runtime_dir = Path(runtime_dir)\n    runtime_dir.mkdir(parents=True, exist_ok=True)\n    env = dict(os.environ)\n    env.update(TOKENIZERS_PARALLELISM="false", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",\n               HF_MODULES_CACHE=str(runtime_dir / "hf_modules"), PYTHONUNBUFFERED="1")\n    target = runtime_dir / "legacy_packages"\n    probe = (\n        "import transformers; "\n        + ("assert transformers.__version__.startswith(\'4.49.\'), transformers.__version__; " if backend == "legacy" else "")\n        + "from transformers import AutoProcessor, AutoTokenizer, CLIPModel; "\n        + ("from transformers import AutoModelForCausalLM; " if backend == "legacy" else\n           "from transformers import Florence2ForConditionalGeneration; ")\n        + "import einops, timm, accelerate; print(transformers.__version__)"\n    )\n    if target.is_dir():\n        env["PYTHONPATH"] = str(target) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")\n    result = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)\n    if result.returncode == 0:\n        print("Caption subprocess Transformers:", result.stdout.strip())\n        return env\n    if backend != "legacy":\n        raise RuntimeError("The local native Florence checkpoint cannot load in this runtime. "\n                           "Use a matching native Transformers installation or the attached legacy model + offline 4.49 wheels.\\n"\n                           + result.stderr[-3000:])\n    wheel_dirs = offline_caption_wheels([Path(asset_root), Path("/kaggle/input")], runtime_dir / "wheels")\n    if not wheel_dirs:\n        raise FileNotFoundError(\n            "The attached Microsoft Florence checkpoint needs Transformers 4.49.x. "\n            f"No offline wheels found under {asset_root}. Attach offline_wheels.zip/wheels.zip or a wheel folder "\n            "with transformers==4.49.0, tokenizers 0.21.x and huggingface-hub <1 compatible with this Python version. "\n            "The notebook never enables Internet or modifies the kernel\'s Transformers."\n        )\n    command = [sys.executable, "-m", "pip", "install", "--no-index", "--only-binary=:all:",\n               "--no-deps", "--upgrade", "--target", str(target), "transformers==4.49.0",\n               "tokenizers>=0.21,<0.22", "huggingface-hub>=0.26,<1"]\n    import importlib.util\n    for module in ("einops", "timm", "accelerate"):\n        if importlib.util.find_spec(module) is None:\n            command.append(module)\n    for directory in wheel_dirs:\n        command.extend(["--find-links", directory])\n    print("Preparing isolated Florence dependencies from attached offline wheels; the notebook kernel is unchanged.")\n    installed = subprocess.run(command, env=env, capture_output=True, text=True)\n    if installed.returncode:\n        raise RuntimeError("Offline Florence dependency setup failed. Wheels must match Kaggle\'s Python/platform.\\n"\n                           + installed.stdout[-2000:] + installed.stderr[-3000:])\n    base_pythonpath = os.environ.get("PYTHONPATH", "")\n    env["PYTHONPATH"] = str(target) + (os.pathsep + base_pythonpath if base_pythonpath else "")\n    checked = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)\n    if checked.returncode:\n        raise RuntimeError("The offline Florence environment still lacks a compatible dependency. "\n                           "No kernel packages were changed.\\n" + checked.stderr[-4000:])\n    print("Isolated caption subprocess Transformers:", checked.stdout.strip())\n    return env\n\ndef florence_worker_main(request_path, phase, *, generate_records, encode_texts):\n    import torch\n\n    payload = json.loads(Path(request_path).read_text(encoding="utf-8"))\n    validate_caption_records(payload["records"])\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a CUDA GPU")\n    settings = payload["settings"]\n    if type(payload.get("max_new_images", 0)) is not int or payload.get("max_new_images", 0) < 0:\n        raise ValueError("max_new_images must be a nonnegative integer")\n    for key in ("batch_size", "max_new_tokens"):\n        if type(settings[key]) is not int or settings[key] < 1:\n            raise ValueError(f"{key} must be a positive integer")\n    if type(settings["seed"]) is not int or not 0 <= settings["seed"] < 2**32:\n        raise ValueError("Caption seed must be an integer in [0, 2**32)")\n    torch.manual_seed(settings["seed"])\n    torch.cuda.manual_seed_all(settings["seed"])\n    if phase == "generate":\n        from importlib.metadata import version\n        if not version("transformers").startswith("4.49."):\n            raise RuntimeError("Legacy Florence generation must use the isolated Transformers 4.49 runtime")\n        path, count = validate_florence_prefix_folder(payload["model_path"])\n        protocol = {"schema": 1, "method": METHOD, "model_id": MODEL_ID,\n                    "model_assets": checkpoint_asset_identity(path), "parameters": count,\n                    "prompt": TASK_PROMPT, "decoding": "greedy_beam1_prefix_cleanup_first8",\n                    "dtype": "bfloat16" if torch.cuda.is_bf16_supported() else "float16",\n                    "policy": "isic_visible_attributes_v1",\n                    "images": [{key: row[key] for key in ("image_id", "split", "image_sha256")}\n                               for row in payload["records"]],\n                    "settings": settings, "runtime": runtime_versions(), "code_sha256": payload["code_sha256"]}\n        output = Path(payload["output"])\n        identity = lock_protocol(output / "generation_protocol" / "protocol.json", protocol)\n        if not (output / "status.json").exists() and (output / "captions.csv").exists():\n            raise ValueError("Unregistered captions.csv; use a fresh output root")\n        generate_records(payload, identity)\n    elif phase == "encode":\n        encode_texts(payload)\n    else:\n        raise ValueError("Phase must be generate or encode")\n\nif __name__ == "__main__":\n    florence_worker_main(sys.argv[1], sys.argv[2], generate_records=generate_isic_records,\n                         encode_texts=encode_isic_texts)\n'

caption_records = []
for role, frame in (('train', train_df), ('val', val_df), ('test', test_df)):
    caption_records.extend({'image_id': str(row.image_id), 'image_path': str(row.image_path),
                            'split': role, 'image_sha256': file_sha256(row.image_path)}
                           for row in frame.itertuples(index=False))
validate_caption_records(caption_records)
local_florence, backend = florence_checkpoint(FLORENCE_MODEL_REF, SEARCH_ROOTS)
if backend != 'legacy':
    raise ValueError('Use the original Microsoft Florence-2-base legacy checkpoint for this Prefix control')
florence_path, parameter_count = validate_florence_prefix_folder(local_florence)
clip_path = resolve_clip_model(CLIP_MODEL_LOCAL_PATH, SEARCH_ROOTS)
runtime_dir = OUT_PUT_DIR / 'worker_runtime'
runtime_dir.mkdir(parents=True, exist_ok=True)
# Only this child uses 4.49; offline --target setup does not replace kernel packages.
florence_env = caption_runtime_environment('legacy', OFFLINE_ASSETS_ROOT, runtime_dir)
clip_env = dict(os.environ)
clip_env.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false',
                HF_HUB_DISABLE_TELEMETRY='1', PYTHONUNBUFFERED='1')
clip_probe = subprocess.run([sys.executable, '-c',
    'from transformers import AutoTokenizer, CLIPTextModelWithProjection; import accelerate, torch; print(torch.__version__)'],
    env=clip_env, capture_output=True, text=True)
if clip_probe.returncode:
    raise RuntimeError('The kernel runtime cannot load local CLIP text weights. ' + clip_probe.stderr[-3000:])
worker_path = runtime_dir / 'isic_florence_prefix_worker.py'
worker_path.write_text(ISIC_WORKER_SOURCE, encoding='utf-8')
caption_payload = {'records': caption_records, 'settings': CAPTION_SETTINGS,
    'model_path': str(florence_path), 'clip_path': str(clip_path), 'output': str(CAPTION_OUTPUT),
    'max_new_images': MAX_NEW_CAPTION_IMAGES, 'code_sha256': object_hash(ISIC_WORKER_SOURCE)}
request_path = runtime_dir / 'request.json'
atomic_json(request_path, caption_payload)
run_worker(worker_path, request_path, 'generate', florence_env, runtime_dir / 'generate.log')
run_worker(worker_path, request_path, 'encode', clip_env, runtime_dir / 'encode.log')
print('Florence and CLIP workers exited; GPU memory is released before segmentation initialization.')


In [ ]:
rows, caption_status = validated_export(caption_payload)
CAPTION_CSV = CAPTION_OUTPUT / 'captions.csv'
text_path = CAPTION_OUTPUT / 'text_cache' / 'clip_text.pt'
text_status = json.loads((CAPTION_OUTPUT / 'text_cache' / 'status.json').read_text(encoding='utf-8'))
if file_sha256(text_path) != text_status['sha256']:
    raise ValueError('CLIP text artifact hash mismatch')
text_cache = torch.load(text_path, map_location='cpu', weights_only=True)
if text_cache['image_ids'] != [r['image_id'] for r in caption_records] or text_cache['caption_csv_sha256'] != caption_status['csv_sha256']:
    raise ValueError('CLIP vectors do not align with the current image IDs / captions.csv')
if text_cache['identity'] != text_status['identity']:
    raise ValueError('CLIP cache protocol mismatch')
validate_text_cache(text_cache, rows, caption_status['csv_sha256'], text_status['identity'])
text_embeddings = text_cache['embeddings'].float().to(DEVICE)
text_valid = text_cache['valid'].bool().to(DEVICE)
if not torch.isfinite(text_embeddings).all() or torch.count_nonzero(text_embeddings[~text_valid]):
    raise ValueError('Invalid/nonfinite text embeddings; rejected captions must have zero vectors')
TEXT_DIM = text_embeddings.shape[1]
TEXT_POSITION = {image_id: pos for pos, image_id in enumerate(text_cache['image_ids'])}
caption_by_id = {row['image_id']: row['final_caption'] for row in rows}
candidate_text_positions = torch.tensor([TEXT_POSITION[str(x)] for x in candidate_df.image_id], device=DEVICE)
candidate_text_bank = text_embeddings[candidate_text_positions]
candidate_text_valid = text_valid[candidate_text_positions]

def query_text_inputs(image_ids):
    positions = torch.tensor([TEXT_POSITION[str(image_id)] for image_id in image_ids], device=DEVICE)
    return text_embeddings[positions], text_valid[positions]

coverage = []
for role, frame in (('train', train_df), ('val', val_df), ('test', test_df), ('candidate_pool', candidate_df)):
    positions = torch.tensor([TEXT_POSITION[str(x)] for x in frame.image_id], device=DEVICE)
    coverage.append({'role': role, 'images': len(frame), 'usable_captions': int(text_valid[positions].sum()),
                     'usable_rate': float(text_valid[positions].float().mean())})
pd.DataFrame(coverage).to_csv(OUT_PUT_DIR / 'caption_coverage.csv', index=False)
display(pd.DataFrame(coverage))
print('Caption CSV SHA256:', caption_status['csv_sha256'])
print('CLIP text vectors:', tuple(text_embeddings.shape), '| policy:', POLICY)
del text_cache


## Full-Both Segmentation Model

Query CLIP text is projected and gated into each query patch token. The selected candidate caption is independently
projected/gated into context tokens, before the same visual cross-attention + DPT-lite decoder.
Selector score = original visual score + sigmoid(learned mix) * valid query/candidate CLIP text cosine.
An invalid query/candidate text pair contributes zero semantic score. Self candidates are masked after score fusion.
The segmentation-loss path uses straight-through weights for **both** selected image tokens and candidate text vectors;
the policy path samples hard actions from the same masked scores. Its reward compares query-image+query-text against
that same input plus selected candidate-image+candidate-text. Captions are never treated as ground truth masks.


In [ ]:
def resolve_dinov3_weights():
    path = Path(DINOV3_WEIGHTS)
    if not path.is_file():
        raise FileNotFoundError(f"Khong tim thay DINOv3 weights: {path}")
    return str(path)


def load_dinov3_model(repo_dir, model_name, weights_path):
    repo_dir = Path(repo_dir)
    if not repo_dir.is_dir():
        raise FileNotFoundError(f"Khong tim thay DINOV3_REPO_DIR: {repo_dir}")
    weights_path = Path(weights_path)
    if not weights_path.is_file():
        raise FileNotFoundError(f"Khong tim thay DINOV3_WEIGHTS: {weights_path}")
    return torch.hub.load(str(repo_dir), model_name, source="local", weights=str(weights_path))


def ensure_dinov3_assets():
    resolve_dinov3_weights()
    if not Path(DINOV3_REPO_DIR).is_dir():
        raise FileNotFoundError(f"Khong tim thay DINOv3 repo: {DINOV3_REPO_DIR}")
    print("DINOv3 repo:", DINOV3_REPO_DIR)
    print("DINOv3 weights:", DINOV3_WEIGHTS)


ensure_dinov3_assets()

class FrozenDinoV3FeatureExtractor(nn.Module):
    def __init__(self, repo_dir, model_name, weights_path):
        super().__init__()
        self.backbone = load_dinov3_model(repo_dir, model_name, weights_path)
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        self.backbone.eval()

    def forward_features(self, images):
        with torch.no_grad():
            with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                features = self.backbone.forward_features(images)
        if isinstance(features, dict):
            if "x_norm_patchtokens" in features:
                patch_tokens = features["x_norm_patchtokens"]
            elif "x_norm_patch_tokens" in features:
                patch_tokens = features["x_norm_patch_tokens"]
            else:
                tensor_values = [v for v in features.values() if torch.is_tensor(v) and v.ndim == 3]
                if not tensor_values:
                    raise KeyError(f"Khong tim thay patch tokens trong keys: {list(features)}")
                patch_tokens = tensor_values[0]

            if "x_norm_clstoken" in features:
                global_features = features["x_norm_clstoken"]
            elif "x_norm_cls_token" in features:
                global_features = features["x_norm_cls_token"]
            else:
                global_features = patch_tokens.mean(dim=1)
            return patch_tokens.float(), global_features.float()

        if torch.is_tensor(features):
            if features.ndim == 3:
                return features[:, 1:].float(), features[:, 0].float()
            if features.ndim == 2:
                raise RuntimeError("Backbone chi tra ve global feature, khong co patch tokens.")
        raise TypeError(f"Output DINO khong ho tro: {type(features)}")


class TACSSelector(nn.Module):
    def __init__(self, feature_dim=384, hidden_dim=384, dropout=0.1, temperature=1.0):
        super().__init__()
        self.temperature = float(temperature)
        self.query_proj = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, feature_dim),
        )
        self.candidate_proj = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, feature_dim),
        )
        self.logit_scale = nn.Parameter(torch.tensor(1.0))

    def score(self, query_features, candidate_features):
        q = F.normalize(self.query_proj(query_features.float()), dim=-1)
        c = F.normalize(self.candidate_proj(candidate_features.float()), dim=-1)
        scores = q @ c.T
        return scores * self.logit_scale.clamp(0.01, 100.0) / max(self.temperature, 1e-6)


class GatedCrossAttentionBlock(nn.Module):
    def __init__(self, feature_dim=384, num_heads=6, dropout=0.1):
        super().__init__()
        self.query_norm = nn.LayerNorm(feature_dim)
        self.context_norm = nn.LayerNorm(feature_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=feature_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.gate = nn.Sequential(
            nn.LayerNorm(feature_dim * 2),
            nn.Linear(feature_dim * 2, feature_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feature_dim, feature_dim),
            nn.Sigmoid(),
        )
        self.ffn = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, feature_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feature_dim * 4, feature_dim),
        )

    def forward(self, query_tokens, context_tokens):
        q = self.query_norm(query_tokens)
        c = self.context_norm(context_tokens)
        attn_out, _ = self.attn(q, c, c, need_weights=False)
        gate_input = torch.cat([query_tokens, attn_out], dim=-1)
        gate = self.gate(gate_input)
        x = query_tokens + gate * attn_out
        x = x + self.ffn(x)
        return x


class DPTLiteHead(nn.Module):
    def __init__(self, feature_dim=384, out_channels=1, dropout=0.1):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(feature_dim),
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(feature_dim, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.Dropout2d(dropout),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),

            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),

            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),

            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),

            nn.Conv2d(32, out_channels, kernel_size=1),
        )

    def forward(self, tokens, image_size=256):
        b, n, d = tokens.shape
        grid = int(math.sqrt(n))
        if grid * grid != n:
            raise RuntimeError(f"So patch tokens {n} khong tao duoc grid vuong.")
        x = self.proj(tokens)
        x = x.transpose(1, 2).reshape(b, d, grid, grid)
        logits = self.decoder(x)
        if logits.shape[-2:] != (image_size, image_size):
            logits = F.interpolate(logits, size=(image_size, image_size), mode="bilinear", align_corners=False)
        return logits


class TACSSegmentationModel(nn.Module):
    def __init__(self, repo_dir, model_name, weights_path, feature_dim=384):
        super().__init__()
        self.feature_extractor = FrozenDinoV3FeatureExtractor(repo_dir, model_name, weights_path)
        self.selector = TACSSelector(
            feature_dim=feature_dim,
            hidden_dim=feature_dim,
            dropout=TACS_SELECTOR_DROPOUT,
            temperature=TACS_SELECTOR_TEMPERATURE,
        )
        self.cross_attention = nn.ModuleList([
            GatedCrossAttentionBlock(
                feature_dim=feature_dim,
                num_heads=CROSS_ATTENTION_HEADS,
                dropout=DECODER_DROPOUT,
            )
            for _ in range(NUM_CROSS_ATTENTION_LAYERS)
        ])
        self.decoder = DPTLiteHead(feature_dim=feature_dim, dropout=DECODER_DROPOUT)

    def encode_images(self, images):
        return self.feature_extractor.forward_features(images)

    def select_context(self, query_global, candidate_global_bank, use_gumbel=True, forbidden_indices=None):
        selector_scores = self.selector.score(query_global, candidate_global_bank)
        if forbidden_indices is not None:
            valid = forbidden_indices >= 0
            if torch.any(valid):
                selector_scores = selector_scores.clone()
                rows = torch.arange(selector_scores.size(0), device=selector_scores.device)[valid]
                selector_scores[rows, forbidden_indices[valid]] = -1e4
        if self.training and use_gumbel:
            selection_weights = F.gumbel_softmax(
                selector_scores,
                tau=TACS_GUMBEL_TAU,
                hard=True,
                dim=1,
            )
            selected_indices = selection_weights.argmax(dim=1)
        else:
            selected_indices = selector_scores.argmax(dim=1)
            selection_weights = F.one_hot(selected_indices, num_classes=selector_scores.size(1)).to(selector_scores.dtype)
        entropy = torch.distributions.Categorical(logits=selector_scores.float()).entropy().mean()
        return {
            "selector_scores": selector_scores,
            "selection_weights": selection_weights,
            "selected_indices": selected_indices,
            "selector_entropy": entropy,
        }

    def decode_with_context_tokens(self, query_tokens, context_tokens):
        x = query_tokens
        for block in self.cross_attention:
            x = block(x, context_tokens)
        return self.decoder(x, image_size=IMAGE_SIZE)

    def forward_query_only_from_tokens(self, query_tokens):
        return self.decoder(query_tokens, image_size=IMAGE_SIZE)

    def forward_with_context(self, images, context_tokens):
        query_tokens, query_global = self.encode_images(images)
        logits = self.decode_with_context_tokens(query_tokens, context_tokens)
        return logits, query_tokens, query_global


"""Text-conditioned selector and dense feature fusion for the ISIC TACS variant."""
import torch
from torch import nn
import torch.nn.functional as F


def text_for_indices(weights, embeddings, valid):
    # Straight-through weights must remain attached on the segmentation-loss path.
    weights = weights.float()
    return weights @ embeddings.float(), weights @ valid.float()


def make_isic_full_both_class(base_class):
    class MultimodalISICTACS(base_class):
        def __init__(self, *args, text_dim=512, **kwargs):
            super().__init__(*args, **kwargs)
            feature_dim = kwargs.get("feature_dim", 384)
            self.query_text_projection = nn.Sequential(nn.LayerNorm(text_dim), nn.Linear(text_dim, feature_dim), nn.GELU())
            self.candidate_text_projection = nn.Sequential(nn.LayerNorm(text_dim), nn.Linear(text_dim, feature_dim), nn.GELU())
            self.query_text_gate = nn.Linear(feature_dim * 2, 1)
            self.candidate_text_gate = nn.Linear(feature_dim * 2, 1)
            nn.init.zeros_(self.query_text_gate.weight)
            nn.init.zeros_(self.candidate_text_gate.weight)
            nn.init.constant_(self.query_text_gate.bias, -4.0)
            nn.init.constant_(self.candidate_text_gate.bias, -4.0)
            self.semantic_mix_logit = nn.Parameter(torch.tensor(0.0))

        def select_full_both(self, query_global, candidate_global, q_text, q_valid, c_bank, c_valid,
                             forbidden_indices=None, sample=False, tau=0.1):
            scores = self.selector.score(query_global, candidate_global).float()
            semantic = F.normalize(q_text.float(), dim=1) @ F.normalize(c_bank.float(), dim=1).T
            valid = q_valid.bool()[:, None] & c_valid.bool()[None, :]
            scores = scores + torch.sigmoid(self.semantic_mix_logit) * semantic * valid
            if forbidden_indices is not None:
                forbidden_indices = forbidden_indices.to(scores.device)
                in_pool = forbidden_indices >= 0
                scores = scores.clone()
                scores[torch.arange(len(scores), device=scores.device)[in_pool], forbidden_indices[in_pool]] = -torch.inf
            if not torch.isfinite(scores).any(dim=1).all():
                raise ValueError("No candidate remains after self-exclusion")
            if sample:
                weights = F.gumbel_softmax(scores, tau=tau, hard=True, dim=1)
            else:
                weights = F.one_hot(scores.argmax(1), scores.shape[1]).to(scores.dtype)
            return {"selector_scores": scores, "selection_weights": weights,
                    "selected_indices": weights.argmax(1),
                    "selector_entropy": torch.distributions.Categorical(logits=scores).entropy().mean()}

        def fuse_text(self, tokens, text, valid, candidate=False):
            projection = self.candidate_text_projection if candidate else self.query_text_projection
            gate_layer = self.candidate_text_gate if candidate else self.query_text_gate
            features = projection(text.float())[:, None, :].expand(-1, tokens.shape[1], -1)
            gate = torch.sigmoid(gate_layer(torch.cat([tokens.float(), features], dim=-1)))
            gate = gate * valid.to(gate.dtype)[:, None, None]
            return tokens.float() + gate * features, gate.mean(dim=(1, 2))

        def decode_full_both(self, query_tokens, context_tokens, q_text, q_valid, c_text, c_valid):
            query, q_gate = self.fuse_text(query_tokens, q_text, q_valid)
            context, c_gate = self.fuse_text(context_tokens, c_text, c_valid, candidate=True)
            logits = self.decode_with_context_tokens(query, context)
            return logits, q_gate, c_gate

        def query_only_with_text(self, query_tokens, q_text, q_valid):
            query, _ = self.fuse_text(query_tokens, q_text, q_valid)
            return self.forward_query_only_from_tokens(query)

    return MultimodalISICTACS

FullBothISICModel = make_isic_full_both_class(TACSSegmentationModel)
model = FullBothISICModel(DINOV3_REPO_DIR, DINOV3_MODEL_NAME, resolve_dinov3_weights(),
                          feature_dim=FEATURE_DIM, text_dim=TEXT_DIM).to(DEVICE)
print('Full-both segmentation trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
@torch.no_grad()
def build_candidate_banks(network, annotations):
    dataset = ISICSegmentationDataset(annotations, eval_transform, return_mask=False)
    loader = DataLoader(
        dataset,
        batch_size=CANDIDATE_FEATURE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )
    patch_banks = []
    global_banks = []
    fnames = []
    image_ids = []
    network.eval()
    for batch in tqdm(loader, desc="encode candidate pool", leave=False):
        images = batch["image"].to(DEVICE, non_blocking=True)
        patch_tokens, global_features = network.encode_images(images)
        patch_banks.append(patch_tokens.detach().cpu().half())
        global_banks.append(global_features.detach().cpu().float())
        fnames.extend(batch["fname"])
        image_ids.extend(batch["image_id"])

    candidate_patch_bank = torch.cat(patch_banks, dim=0).to(DEVICE, non_blocking=True)
    candidate_global_bank = torch.cat(global_banks, dim=0).to(DEVICE, non_blocking=True)
    return candidate_patch_bank, candidate_global_bank, fnames, image_ids


candidate_patch_bank, candidate_global_bank, candidate_fnames, candidate_image_ids = build_candidate_banks(model, candidate_df)
torch.save({
    "candidate_patch_bank": candidate_patch_bank.detach().cpu(),
    "candidate_global_bank": candidate_global_bank.detach().cpu(),
    "candidate_fnames": candidate_fnames,
    "candidate_image_ids": candidate_image_ids,
    "candidate_pool_positions": candidate_pool_positions,
    "seed": SEED,
}, OUT_PUT_DIR / "candidate_banks.pt")

print("candidate_patch_bank:", tuple(candidate_patch_bank.shape), candidate_patch_bank.dtype, candidate_patch_bank.device)
print("candidate_global_bank:", tuple(candidate_global_bank.shape), candidate_global_bank.dtype, candidate_global_bank.device)


In [ ]:
def segmentation_loss_per_sample(logits, masks):
    bce = F.binary_cross_entropy_with_logits(logits, masks, reduction="none").mean(dim=(1, 2, 3))
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    intersection = (probs * masks).sum(dim=dims)
    union = probs.sum(dim=dims) + masks.sum(dim=dims)
    dice_loss = 1.0 - (2.0 * intersection + 1e-6) / (union + 1e-6)
    return BCE_LOSS_WEIGHT * bce + DICE_LOSS_WEIGHT * dice_loss


def dice_iou_from_logits(logits, masks, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    dims = (1, 2, 3)
    intersection = (preds * masks).sum(dim=dims)
    pred_sum = preds.sum(dim=dims)
    mask_sum = masks.sum(dim=dims)
    dice = (2.0 * intersection + 1e-6) / (pred_sum + mask_sum + 1e-6)
    iou = (intersection + 1e-6) / (pred_sum + mask_sum - intersection + 1e-6)
    return dice, iou


def soft_dice_iou_from_logits(logits, masks):
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    intersection = (probs * masks).sum(dim=dims)
    pred_sum = probs.sum(dim=dims)
    mask_sum = masks.sum(dim=dims)
    dice = (2.0 * intersection + 1e-6) / (pred_sum + mask_sum + 1e-6)
    iou = (intersection + 1e-6) / (pred_sum + mask_sum - intersection + 1e-6)
    return dice, iou


candidate_row_id_to_pool_pos = {
    int(row_id): int(pos)
    for pos, row_id in enumerate(candidate_df["row_id"].tolist())
}


def forbidden_pool_indices_from_row_ids(row_ids):
    values = [
        candidate_row_id_to_pool_pos.get(int(row_id), -1)
        for row_id in row_ids.detach().cpu().tolist()
    ]
    return torch.tensor(values, dtype=torch.long, device=DEVICE)


def select_candidate_tokens_from_weights(selection_weights):
    flat_bank = candidate_patch_bank.flatten(1)
    weights = selection_weights.to(device=flat_bank.device, dtype=flat_bank.dtype)
    selected_flat = weights @ flat_bank
    return selected_flat.view(weights.size(0), *candidate_patch_bank.shape[1:]).float()


def gather_candidate_tokens(indices):
    return candidate_patch_bank[indices.detach().to(candidate_patch_bank.device)].float()


train_dataset = ISICSegmentationDataset(train_df, train_transform, return_mask=True)
val_dataset = ISICSegmentationDataset(val_df, eval_transform, return_mask=True)
test_dataset = ISICSegmentationDataset(test_df, eval_transform, return_mask=True)

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
}
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = 2

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=train_generator, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

warmup_epochs = max(1, min(WARMUP_EPOCHS, EPOCHS))
main_epochs = max(1, EPOCHS - warmup_epochs)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.2,
    end_factor=1.0,
    total_iters=warmup_epochs,
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=main_epochs,
    eta_min=1e-7,
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_epochs],
)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

checkpoint_path = OUT_PUT_DIR / "final_tacs_isic2017_segmentation.pt"
last_checkpoint_path = OUT_PUT_DIR / "checkpoint_last.pt"
best_checkpoint_path = OUT_PUT_DIR / "best_val_checkpoint.pt"

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))


In [ ]:
TRAINING_CODE_SHA256 = '1c42a522b50bb328d94e87a199e9f763fba7f7216d3294db64fcf8a28778b4d7'
BASELINE_DICE = 0.856
BASELINE_IOU = 0.7738

import importlib.metadata
import gc

split_identity = []
for role, frame in (('train', train_df), ('val', val_df), ('test', test_df)):
    for row in frame.itertuples(index=False):
        split_identity.append({'image_id': str(row.image_id), 'split': role,
                               'image_sha256': file_sha256(row.image_path), 'mask_sha256': file_sha256(row.mask_path)})
protocol = {
    'schema': 1, 'experiment': 'isic2017_florence_prefix_full_both_448', 'seed': SEED,
    'split_identity': split_identity, 'candidate_ids': candidate_df.image_id.astype(str).tolist(),
    'caption_csv_sha256': caption_status['csv_sha256'], 'text_cache_identity': text_status['identity'],
    'dino_weights_sha256': file_sha256(resolve_dinov3_weights()), 'code_sha256': TRAINING_CODE_SHA256,
    'configuration': {key: globals()[key] for key in (
        'IMAGE_SIZE', 'BATCH_SIZE', 'NUM_WORKERS', 'EPOCHS', 'LEARNING_RATE', 'WEIGHT_DECAY',
        'WARMUP_EPOCHS', 'PATIENCE', 'GRAD_CLIP_NORM', 'CANDIDATE_POOL_FRACTION', 'TACS_GUMBEL_TAU',
        'TACS_POLICY_WEIGHT', 'TACS_ENTROPY_WEIGHT', 'TACS_SELECTOR_DROPOUT', 'TACS_SELECTOR_TEMPERATURE',
        'NUM_CROSS_ATTENTION_LAYERS', 'CROSS_ATTENTION_HEADS', 'DECODER_DROPOUT', 'DICE_LOSS_WEIGHT',
        'BCE_LOSS_WEIGHT', 'THRESHOLD')},
    'runtime': {name: importlib.metadata.version(name) for name in ('torch', 'torchvision', 'numpy', 'Pillow')},
    'fusion': 'query_text_to_query_tokens_and_candidate_text_to_context_tokens_v1',
    'selector': 'original_visual_score_plus_sigmoid_mix_times_valid_clip_text_cosine',
    'checkpoint_selection': 'maximum_validation_hard_dice',
}
protocol_path = OUT_PUT_DIR / 'training_protocol' / 'protocol.json'
if not protocol_path.exists() and any(p.exists() for p in (best_checkpoint_path, last_checkpoint_path)):
    raise ValueError('Unregistered checkpoint in output directory; use a new output root')
PROTOCOL_ID = lock_protocol(protocol_path, protocol)
for role, frame in (('train', train_df), ('val', val_df), ('test', test_df), ('candidate_pool', candidate_df)):
    atomic_csv(OUT_PUT_DIR / f'{role}_split.csv', frame.to_dict('records'), list(frame.columns))
print('Locked training protocol:', PROTOCOL_ID)

def atomic_checkpoint(path, value):
    temporary = Path(path).with_suffix('.pt.tmp')
    torch.save(value, temporary)
    temporary.replace(path)

def move_batch_to_device(batch):
    return {'images': batch['image'].to(DEVICE, non_blocking=True),
            'masks': batch['mask'].to(DEVICE, non_blocking=True),
            'fname': batch['fname'], 'image_id': batch['image_id'],
            'row_id': batch['row_id'].to(DEVICE, non_blocking=True)}

def full_both_selection(query_global, q_text, q_valid, forbidden=None, training=False):
    return model.select_full_both(query_global, candidate_global_bank, q_text, q_valid,
        candidate_text_bank, candidate_text_valid, forbidden_indices=forbidden,
        sample=training, tau=TACS_GUMBEL_TAU)

@torch.no_grad()
def evaluate_full_both(loader, desc='val', return_rows=False):
    model.eval()
    totals = {key: 0.0 for key in ('loss', 'dice', 'iou', 'soft_dice', 'soft_iou', 'q_gate', 'c_gate',
                                   'query_caption_rate', 'candidate_caption_rate')}
    count, predictions = 0, []
    for batch in tqdm(loader, desc=desc, leave=False):
        moved = move_batch_to_device(batch)
        q_text, q_valid = query_text_inputs(moved['image_id'])
        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            q_tokens, q_global = model.encode_images(moved['images'])
            details = full_both_selection(q_global, q_text, q_valid)
            indices = details['selected_indices']
            context = gather_candidate_tokens(indices)
            logits, q_gate, c_gate = model.decode_full_both(q_tokens, context, q_text, q_valid,
                candidate_text_bank[indices], candidate_text_valid[indices])
            losses = segmentation_loss_per_sample(logits, moved['masks'])
        dice, iou = dice_iou_from_logits(logits.float(), moved['masks'], threshold=THRESHOLD)
        soft_dice, soft_iou = soft_dice_iou_from_logits(logits.float(), moved['masks'])
        values = {'loss': losses, 'dice': dice, 'iou': iou, 'soft_dice': soft_dice, 'soft_iou': soft_iou,
                  'q_gate': q_gate, 'c_gate': c_gate, 'query_caption_rate': q_valid.float(),
                  'candidate_caption_rate': candidate_text_valid[indices].float()}
        for key, value in values.items():
            totals[key] += value.float().sum().item()
        count += len(dice)
        if return_rows:
            for j, image_id in enumerate(moved['image_id']):
                pos = int(indices[j])
                context_id = str(candidate_df.iloc[pos].image_id)
                predictions.append({'image_id': str(image_id), 'fname': moved['fname'][j],
                    'context_pool_position': pos, 'context_image_id': context_id,
                    'query_caption': caption_by_id[str(image_id)], 'query_caption_valid': bool(q_valid[j]),
                    'candidate_caption': caption_by_id[context_id], 'candidate_caption_valid': bool(candidate_text_valid[pos]),
                    **{key: float(value[j]) for key, value in values.items()}})
    if count != len(loader.dataset):
        raise ValueError('Evaluation did not cover every requested image')
    return {key: value / count for key, value in totals.items()}, pd.DataFrame(predictions)


In [ ]:
def train_full_both_epoch(epoch):
    model.train()
    model.feature_extractor.eval()
    totals = {key: 0.0 for key in ('loss', 'dice', 'iou', 'reward', 'policy_loss', 'q_gate', 'c_gate')}
    count = 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch:03d} [train]', leave=False):
        moved = move_batch_to_device(batch)
        q_text, q_valid = query_text_inputs(moved['image_id'])
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            q_tokens, q_global = model.encode_images(moved['images'])
            forbidden = forbidden_pool_indices_from_row_ids(moved['row_id'])
            details = full_both_selection(q_global, q_text, q_valid, forbidden, training=True)
            context = select_candidate_tokens_from_weights(details['selection_weights'])
            c_text, c_valid = text_for_indices(details['selection_weights'], candidate_text_bank, candidate_text_valid)
            logits, q_gate, c_gate = model.decode_full_both(q_tokens, context, q_text, q_valid, c_text, c_valid)
            grad_loss = segmentation_loss_per_sample(logits, moved['masks']).mean()
            policy = torch.distributions.Categorical(logits=details['selector_scores'].float())
            actions = policy.sample()
            with torch.no_grad():
                # Keep the query caption in BOTH reward terms; only the retrieved image/text is added.
                no_context = model.query_only_with_text(q_tokens, q_text, q_valid)
                policy_logits, _, _ = model.decode_full_both(q_tokens, gather_candidate_tokens(actions),
                    q_text, q_valid, candidate_text_bank[actions], candidate_text_valid[actions])
                rewards = (segmentation_loss_per_sample(no_context.float(), moved['masks']) -
                           segmentation_loss_per_sample(policy_logits.float(), moved['masks'])).detach()
                advantages = ((rewards - rewards.mean()) / rewards.std(unbiased=False).clamp_min(1e-6)).detach()
            policy_loss = -(policy.log_prob(actions) * advantages).mean()
            loss = grad_loss + TACS_POLICY_WEIGHT * policy_loss - TACS_ENTROPY_WEIGHT * policy.entropy().mean()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        dice, iou = dice_iou_from_logits(logits.detach().float(), moved['masks'], threshold=THRESHOLD)
        size = len(dice)
        for key, value in {'loss': loss, 'policy_loss': policy_loss, 'reward': rewards.mean(),
                           'dice': dice.mean(), 'iou': iou.mean(), 'q_gate': q_gate.mean(), 'c_gate': c_gate.mean()}.items():
            totals[key] += float(value.detach()) * size
        count += size
    return {key: value / count for key, value in totals.items()}

def checkpoint_state(epoch, history, best_dice, best_epoch, bad_epochs):
    return {'protocol_id': PROTOCOL_ID, 'epoch': epoch, 'history': history, 'best_dice': best_dice,
            'best_epoch': best_epoch, 'bad_epochs': bad_epochs, 'model_state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
            'rng_torch': torch.get_rng_state(), 'rng_cuda': torch.cuda.get_rng_state_all(),
            'rng_numpy': np.random.get_state(), 'rng_python': random.getstate(),
            'rng_loader': train_generator.get_state()}

history, best_val_dice, best_epoch, bad_epochs, start_epoch = [], -1.0, 0, 0, 1
if last_checkpoint_path.is_file():
    if not RESUME_TRAINING:
        raise ValueError('Existing run found; enable RESUME_TRAINING or use a new output root')
    state = torch.load(last_checkpoint_path, map_location='cpu', weights_only=False)
    if state['protocol_id'] != PROTOCOL_ID:
        raise ValueError('Checkpoint protocol does not match this run')
    model.load_state_dict(state['model_state_dict'])
    optimizer.load_state_dict(state['optimizer'])
    scheduler.load_state_dict(state['scheduler'])
    scaler.load_state_dict(state['scaler'])
    history, best_val_dice, best_epoch, bad_epochs = state['history'], state['best_dice'], state['best_epoch'], state['bad_epochs']
    start_epoch = state['epoch'] + 1
    torch.set_rng_state(state['rng_torch'])
    torch.cuda.set_rng_state_all(state['rng_cuda'])
    np.random.set_state(state['rng_numpy'])
    random.setstate(state['rng_python'])
    train_generator.set_state(state['rng_loader'])
    del state
    print('Resuming at epoch', start_epoch, '| best epoch', best_epoch)
completion_path = OUT_PUT_DIR / 'completed.json'
if not completion_path.is_file() and bad_epochs < PATIENCE:
    for epoch in range(start_epoch, EPOCHS + 1):
        train_metrics = train_full_both_epoch(epoch)
        val_metrics, _ = evaluate_full_both(val_loader, desc=f'Epoch {epoch:03d} [val]')
        scheduler.step()
        history.append({'epoch': epoch, **{'train_' + k: v for k, v in train_metrics.items()},
                        **{'val_' + k: v for k, v in val_metrics.items()}, 'lr': optimizer.param_groups[0]['lr']})
        improved = val_metrics['dice'] > best_val_dice
        if improved:
            best_val_dice, best_epoch, bad_epochs = val_metrics['dice'], epoch, 0
        else:
            bad_epochs += 1
        state = checkpoint_state(epoch, history, best_val_dice, best_epoch, bad_epochs)
        if improved:
            atomic_checkpoint(best_checkpoint_path, {'protocol_id': PROTOCOL_ID, 'epoch': epoch,
                'model_state_dict': model.state_dict(), 'val_metrics': val_metrics})
        atomic_checkpoint(last_checkpoint_path, state)
        del state
        pd.DataFrame(history).to_csv(OUT_PUT_DIR / 'training_history.csv', index=False)
        print(f"full_both epoch={epoch:03d} val_dice={val_metrics['dice']:.4f} val_iou={val_metrics['iou']:.4f} "
              f"q_gate={val_metrics['q_gate']:.4f} c_gate={val_metrics['c_gate']:.4f} "
              f"semantic_mix={float(torch.sigmoid(model.semantic_mix_logit).detach()):.4f} best={best_epoch:03d} bad={bad_epochs}/{PATIENCE}", flush=True)
        if bad_epochs >= PATIENCE:
            print('Early stopping at epoch', epoch)
            break
if not history or (history[-1]['epoch'] < EPOCHS and bad_epochs < PATIENCE):
    raise RuntimeError('Training is incomplete; resume before final test')
atomic_json(completion_path, {'protocol_id': PROTOCOL_ID, 'epochs_observed': int(history[-1]['epoch']),
    'epoch_limit': EPOCHS, 'best_epoch': best_epoch, 'best_val_dice': best_val_dice,
    'stopping': 'early_stopping' if bad_epochs >= PATIENCE else 'epoch_limit'})
display(pd.DataFrame(history).tail())


## Locked Test and Historical Comparisons

Finish training/early stopping before test. Use the best validation-Dice checkpoint once and reuse locked test exports.
The table includes earlier displayed outputs: visual Dice 85.60%, IoU 77.38%; Qwen8B Dice 84.8689%, IoU 76.5407%.
Both are historical, rounded stdout references, **not rerun paired controls or multi-seed means**.
Florence results are measured by this run only; no Florence test score is pre-filled.

Keep captions, protocols, caches and checkpoints under this experiment's independent output root. Do not reuse a
Qwen/visual checkpoint here. Resume restores optimizer/scheduler/scaler/RNG state, but augmentation workers can prevent
bitwise equivalence to uninterrupted training. Do not tune the method from these test outcomes.


In [ ]:
completed = json.loads((OUT_PUT_DIR / 'completed.json').read_text(encoding='utf-8'))
if completed['protocol_id'] != PROTOCOL_ID:
    raise ValueError('Wrong completed run')
metrics_path = OUT_PUT_DIR / 'metrics.json'
if metrics_path.is_file():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    if metrics['protocol_id'] != PROTOCOL_ID or metrics['best_epoch'] != completed['best_epoch']:
        raise ValueError('Stale test result; use a separate experiment root')
    print('Reusing locked test result:', metrics_path)
else:
    state = torch.load(best_checkpoint_path, map_location='cpu', weights_only=False)
    if state['protocol_id'] != PROTOCOL_ID or state['epoch'] != completed['best_epoch']:
        raise ValueError('Wrong best checkpoint')
    model.load_state_dict(state['model_state_dict'])
    del state
    # No test masks or scores were used by caption generation, retrieval, training or checkpoint choice.
    test_metrics, test_rows = evaluate_full_both(test_loader, desc='test full_both', return_rows=True)
    test_rows.to_csv(OUT_PUT_DIR / 'test_predictions.csv', index=False)
    metrics = {'protocol_id': PROTOCOL_ID, 'method': 'ISIC Florence-Prefix full_both 448',
        'caption_csv_sha256': caption_status['csv_sha256'], 'seed': SEED, 'image_size': IMAGE_SIZE,
        'train_size': len(train_df), 'val_size': len(val_df), 'test_size': len(test_df),
        **completed, **{'test_' + key: value for key, value in test_metrics.items()},
        'selector_semantic_mix': float(torch.sigmoid(model.semantic_mix_logit).detach())}
    atomic_json(metrics_path, metrics)
print(f"test_dice: {metrics['test_dice']:.6f} ({metrics['test_dice'] * 100:.2f}%)")
print(f"test_iou: {metrics['test_iou']:.6f} ({metrics['test_iou'] * 100:.2f}%)")
comparison = pd.DataFrame([
    {'method': 'Qwen3-VL-8B full_both 448, historical seed42', 'test_dice': 0.848689,
     'test_iou': 0.765407, 'new_run': False},
    {'method': 'Visual TACS variant 448, historical seed42', 'test_dice': BASELINE_DICE,
     'test_iou': BASELINE_IOU, 'new_run': False},
    {'method': 'Florence-Prefix full_both 448', 'test_dice': metrics['test_dice'],
     'test_iou': metrics['test_iou'], 'new_run': True},
])
comparison['dice_delta_vs_qwen8b_pp'] = (comparison.test_dice - 0.848689) * 100
comparison['iou_delta_vs_qwen8b_pp'] = (comparison.test_iou - 0.765407) * 100
comparison['dice_delta_percentage_points'] = (comparison.test_dice - BASELINE_DICE) * 100
comparison['iou_delta_percentage_points'] = (comparison.test_iou - BASELINE_IOU) * 100
comparison.to_csv(OUT_PUT_DIR / 'segmentation_comparison.csv', index=False)
display(comparison)
print('Historical reference is rounded stdout, not a newly rerun baseline or a three-seed mean.')

history_df = pd.read_csv(OUT_PUT_DIR / 'training_history.csv')
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for key, ax in zip(('dice', 'iou', 'loss'), axes):
    ax.plot(history_df.epoch, history_df['train_' + key], label='train')
    ax.plot(history_df.epoch, history_df['val_' + key], label='val')
    ax.axvline(completed['best_epoch'], color='gray', linestyle='--')
    ax.set_title(key)
    ax.set_xlabel('Epoch')
    ax.legend()
fig.tight_layout()
fig.savefig(OUT_PUT_DIR / 'learning_curves.png', dpi=160)
plt.show()


In [ ]:
import textwrap
state = torch.load(best_checkpoint_path, map_location='cpu', weights_only=False)
if state['protocol_id'] != PROTOCOL_ID:
    raise ValueError('Wrong visualization checkpoint')
model.load_state_dict(state['model_state_dict'])
del state
model.eval()
sample_batch = next(iter(test_loader))
moved = move_batch_to_device(sample_batch)
with torch.inference_mode(), torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
    q_tokens, q_global = model.encode_images(moved['images'])
    q_text, q_valid = query_text_inputs(moved['image_id'])
    details = full_both_selection(q_global, q_text, q_valid)
    indices = details['selected_indices']
    logits, _, _ = model.decode_full_both(q_tokens, gather_candidate_tokens(indices), q_text, q_valid,
        candidate_text_bank[indices], candidate_text_valid[indices])
predicted = (torch.sigmoid(logits.float()) > THRESHOLD).cpu()
count = min(4, len(indices))
fig, axes = plt.subplots(count, 4, figsize=(16, 4.5 * count), squeeze=False)
for j in range(count):
    image_id = str(moved['image_id'][j])
    query_row = test_df.loc[test_df.image_id.eq(image_id)].iloc[0]
    candidate_row = candidate_df.iloc[int(indices[j])]
    with Image.open(query_row.image_path) as im:
        axes[j, 0].imshow(im.convert('RGB'))
    axes[j, 0].set_title(textwrap.fill('Query: ' + caption_by_id[image_id] + (' [masked]' if not q_valid[j] else ''), 40), fontsize=9)
    axes[j, 1].imshow(moved['masks'][j, 0].cpu(), cmap='gray')
    axes[j, 1].set_title('Ground truth mask')
    axes[j, 2].imshow(predicted[j, 0], cmap='gray')
    axes[j, 2].set_title('Predicted mask')
    with Image.open(candidate_row.image_path) as im:
        axes[j, 3].imshow(im.convert('RGB'))
    caption = caption_by_id[str(candidate_row.image_id)]
    axes[j, 3].set_title(textwrap.fill('Context: ' + caption + (' [masked]' if not candidate_text_valid[indices[j]] else ''), 40), fontsize=9)
    for ax in axes[j]:
        ax.axis('off')
fig.tight_layout()
fig.savefig(OUT_PUT_DIR / 'segmentation_examples_with_captions.png', dpi=150)
plt.show()


## Artifacts and Interpretation

Default output: `/kaggle/working/outputs/mtacs_isic2017_448_florence_prefix_full_both/`.
Keep caption_generation/captions.csv, generated/ resume caches, source responses, CLIP text cache, both locked
protocols, caption_coverage.csv, split/pool CSVs, best/last checkpoints, history, metrics.json, test_predictions.csv,
segmentation_comparison.csv, learning curves and query/context/caption/mask examples.

Report as **ISIC2017, 448, Florence-Prefix full_both, seed42**. Captioner/decoder/runtime and resulting masks differ
from Qwen; the segmentation architecture, fusion, selector and training rules remain the same. Compare coverage and
gates as well as Dice/IoU. One run does not establish statistical superiority or general medical caption quality.
The DTD rank motivated this transfer but does not guarantee transfer gains. No new GPU metrics were measured by the builder.

API: [Microsoft Florence-2-base model card](https://huggingface.co/microsoft/Florence-2-base).
The notebook uses the existing offline legacy checkpoint with its matching 4.49 runtime, not an automatic checkpoint conversion.
